In [77]:
import re, collections, requests, html, pycallnumber as pycn
from lxml import etree

marc_rda_relators = {
'''Creates dictionary of relator codes common to MARC and RDA'''
    "ape": "appellee",
    "apl": "appellant",
    "arc": "architect",
    "art": "artist",
    "aup": "audio producer",
    "aus": "screenwriter",
    "aut": "author",
    "bka": "book artist",
    "cad": "casting director",
    "chr": "choreographer",
    "cll": "calligrapher",
    "cmp": "composer",
    "com": "compiler",
    "cou": "court governed",
    "csl": "consultant",
    "ctg": "cartographer",
    "dfd": "defendant",
    "dgc": "degree committee member",
    "dgg": "degree granting institution",
    "dgs": "degree supervisor",
    "drt": "director",
    "dsr": "designer",
    "dte": "dedicatee",
    "dto": "dedicator",
    "edd": "editorial director",
    "enj": "enacting jurisdiction",
    "fmd": "film director",
    "fmk": "filmmaker",
    "fmp": "film producer",
    "his": "host institution",
    "inv": "inventor",
    "isb": "issuing body",
    "ive": "interviewee",
    "ivr": "interviewer",
    "jud": "judge",
    "jug": "jurisdiction governed",
    "lbt": "librettist",
    "lsa": "landscape architect",
    "lyr": "lyricist",
    "med": "medium",
    "orm": "organizer",
    "pht": "photographer",
    "pra": "praeses",
    "prf": "performer",
    "prg": "programmer",
    "prn": "production company",
    "pro": "producer",
    "ptf": "plaintiff",
    "rap": "rapporteur",
    "rcp": "addressee",
    "rdd": "radio director",
    "res": "researcher",
    "rpc": "radio producer",
    "rsp": "respondent",
    "rxa": "remix artist",
    "scl": "sculptor",
    "tld": "television director",
    "tlp": "television producer",
    }

'''Gets xml file and sets tree and root'''
tree = etree.parse(r"C:\Users\yello\OneDrive\Documents\EAD2MARC\EAD2MARC_workzone\MC122_ead3.xml")
root = tree.getroot()

'''Checks that document is EAD3'''
if 'http://ead3.archivists.org/schema/' in root.tag:

    '''Get target <c> tag'''
    # (This portion of code was partially revised utilizing ChatGPT 5.2)
    target_id = input("Enter aspace id of target <c>: ").strip()

    result = root.xpath(
        "//*[starts-with(local-name(), 'c') and @id=$t_id]",
        t_id=target_id
    )
    # TODO: change to <c> tags specifically instead of any tag that starts with c??

    '''Set global variables'''
    c0_raw = result[0]
    c0_print = etree.tostring(c0_raw, pretty_print=True, encoding="unicode")

    vaid_fetch = root.xpath(".//*[local-name()='recordid']")
    if vaid_fetch:
        vaid_raw = vaid_fetch[0]
        vaid_fetch = vaid_raw.xpath(".//*[local-name()='descriptivenote']")
        vaid_clean = vaid_raw.xpath("string()").strip(".")
        vaid_head_list = vaid_raw.xpath(".//*[local-name()='head']")
        if vaid_head_list:
            vaid_head = vaid_head_list[0].xpath("string()").strip(".")
            vaid_clean = vaid_clean.replace(vaid_head, "")
        vaid_clean = html.escape(vaid_clean)

    '''Set global names lists'''
    names_list = c0_raw.xpath(".//*[local-name()='origination']") # (Troubleshot using ChatGPT 5.2)

    all_persnames_list = []
    all_corpnames_list = []
    all_famnames_list = []

    creator_names_list = []
    creator_persnames_list = []
    creator_corpnames_list = []
    creator_famnames_list = []

    source_names_list = []
    source_persnames_list = []
    source_corpnames_list = []
    source_famnames_list = []

    for orig in names_list:
        if orig.attrib["label"].lower() == "creator":
            creator_names_list.append(orig)
        elif orig.attrib["label"].lower() == "source":
            source_names_list.append(orig)

    for orig in names_list:
        if orig.xpath(".//*[local-name()='persname']"):
            all_persnames_list.append(orig)
            if orig.attrib["label"].lower() == "creator":
                creator_persnames_list.append(orig)
            elif orig.attrib["label"].lower() == "source":
                source_persnames_list.append(orig)
        elif orig.xpath(".//*[local-name()='corpname']"):
            all_corpnames_list.append(orig)
            if orig.attrib["label"].lower() == "creator":
                creator_corpnames_list.append(orig)
            elif orig.attrib["label"].lower() == "source":
                source_corpnames_list.append(orig)
        elif orig.xpath(".//*[local-name()='famname']"):
            all_famnames_list.append(orig)
            if orig.attrib["label"].lower() == "creator":
                creator_famnames_list.append(orig)
            elif orig.attrib["label"].lower() == "source":
                source_famnames_list.append(orig)

    print (c0_print)
else:
    '''Returns error message if document is not EAD3'''
    print("Uploaded file must be in EAD3 (not EAD 2002 or any other EAD version). Please upload a new file and try again.")


<c03 xmlns="http://ead3.archivists.org/schema/" id="aspace_28d322fd614f513e6e7d466670000c1e" level="item">
            <did>
              <unittitle>Crystallization, for symphony orchestra</unittitle>
              <unitid localtype="aspace_uri">/repositories/11/archival_objects/1590742</unitid>
              <unitid audience="internal" identifier="IMI 7717" type="Publisher Number">IMI
                7717</unitid>
              <origination label="Creator">
                <persname identifier="no2006032090" relator="cmp" rules="rda" source="lcnaf">
                  <part>Adler, Ayal, 1968-</part>
                </persname>
              </origination>
              <physdescstructured coverage="whole" physdescstructuredtype="spaceoccupied">
                <quantity>1</quantity>
                <unittype>folder(s)</unittype>
                <dimensions>42 cm</dimensions>
              </physdescstructured>
              <physdesc localtype="container_summary">1 score (61 pages)</p

In [78]:
def ead2marc_020(unitid_raw):
    
    '''INDICATORS'''
    '''Indicator 1 is constant (blank)'''
    '''Indicator 2 is constant (blank)'''
    
    '''SUBFIELDS'''
    '''SUBFIELD A'''
    unitid_str = unitid_raw.xpath("string()").strip()
    a_020 = f"""<subfield code="a">{unitid_str}</subfield>"""

    field_020_open = f"""<datafield tag="020" ind1=" " ind2=" ">"""
    field_020_str_nb = field_020_open + a_020 + "</datafield>"
    field_020_xml = etree.fromstring(field_020_str_nb)
    field_020_str = etree.tostring(field_020_xml, pretty_print=True, encoding="unicode")

    print(field_020_str)
    return(field_020_xml)

In [79]:
def ead2marc_022(unitid_raw):
    '''INDICATORS'''
    '''Indicator 1 is constant (blank)'''
    '''Indicator 2 is constant (blank)'''
    
    '''SUBFIELDS'''
    '''SUBFIELD A'''
    unitid_str = unitid_raw.xpath("string()").strip()
    a_022 = f"""<subfield code="a">{unitid_str}</subfield>"""

    field_022_open = f"""<datafield tag="022" ind1=" " ind2=" ">"""
    field_022_str_nb = field_022_open + a_022 + "</datafield>"
    field_022_xml = etree.fromstring(field_022_str_nb)
    field_022_str = etree.tostring(field_022_xml, pretty_print=True, encoding="unicode")

    print(field_022_str)
    return(field_022_xml)

In [80]:
def ead2marc_023(unitid_raw):
    '''INDICATORS'''
    '''Indicator 1'''
    unitid_type = unitid_raw.get("type", "").lower()
    if unitid_type == "issn-h":
        ind1_023 = "1"
    else:
        ind1_023 = "0"
    '''Indicator 2 is constant (blank)'''
    
    '''SUBFIELDS'''
    '''SUBFIELD A'''
    unitid_str = unitid_raw.xpath("string()").strip()
    a_023 = f"""<subfield code="a">{unitid_str}</subfield>"""

    field_023_open = f"""<datafield tag="023" ind1="{ind1_023}" ind2=" ">"""
    field_023_str_nb = field_023_open + a_023 + "</datafield>"
    field_023_xml = etree.fromstring(field_023_str_nb)
    field_023_str = etree.tostring(field_023_xml, pretty_print=True, encoding="unicode")

    print(field_023_str)
    return(field_023_xml)

In [81]:
def ead2marc_024(unitid_raw):
    '''INDICATORS'''
    '''Indicator 1'''
    unitid_type = unitid_raw.get("type", "").lower()
    field_2_024 = ""
    if (unitid_type == "isrc") or ("international standard recording code" in unitid_type):
        ind1_024 = "0"
    elif (unitid_type == "upc") or ("universal product code" in unitid_type):
        ind1_024 = "1"
    elif (unitid_type == "ismn") or ("international standard music" in unitid_type):
        ind1_024 = "2"
    elif (unitid_type == "ean") or ("international article number" in unitid_type):
        ind1_024 = "3"
    elif (unitid_type == "sici") or ("serial item and contribution" in unitid_type):
        ind1_024 = "4"
    elif unitid_type:
        ind1_024 = "7"
        field_2_024 = f"""<subfield code="2">{unitid_type}</subfield>"""
    else:
        ind1_024 = "8"
    '''Indicator 2 is constant (blank)'''
    
    '''SUBFIELDS'''
    '''SUBFIELD A'''
    unitid_str = unitid_raw.xpath("string()").strip()
    a_024 = f"""<subfield code="a">{unitid_str}</subfield>"""

    field_024_open = f"""<datafield tag="024" ind1="{ind1_024}" ind2=" ">"""
    field_024_str_nb = field_024_open + a_024 + field_2_024 + "</datafield>"
    field_024_xml = etree.fromstring(field_024_str_nb)
    field_024_str = etree.tostring(field_024_xml, pretty_print=True, encoding="unicode")

    print(field_024_str)
    return(field_024_xml)

In [82]:
def ead2marc_026(unitid_raw):
    '''INDICATORS'''
    '''Indicator 1 is constant (blank)'''
    '''Indicator 2 is constant (blank)'''
    
    '''SUBFIELDS'''
    '''SUBFIELD E'''
    unitid_str = unitid_raw.xpath("string()").strip()
    e_026 = f"""<subfield code="e">{unitid_str}</subfield>"""
    #NOTE: Only unparsed fingerprint identifiers (subfield E) are currently supported

    field_026_open = f"""<datafield tag="026" ind1=" " ind2=" ">"""
    field_026_str_nb = field_026_open + e_026 + "</datafield>"
    field_026_xml = etree.fromstring(field_026_str_nb)
    field_026_str = etree.tostring(field_026_xml, pretty_print=True, encoding="unicode")

    print(field_026_str)
    return(field_026_xml)

In [83]:
def ead2marc_027(unitid_raw):
    '''INDICATORS'''
    '''Indicator 1 is constant (blank)'''
    '''Indicator 2 is constant (blank)'''
    
    '''SUBFIELDS'''
    '''SUBFIELD A'''
    unitid_str = unitid_raw.xpath("string()").strip()
    a_027 = f"""<subfield code="a">{unitid_str}</subfield>"""

    field_027_open = f"""<datafield tag="027" ind1=" " ind2=" ">"""
    field_027_str_nb = field_027_open + a_027 + "</datafield>"
    field_027_xml = etree.fromstring(field_027_str_nb)
    field_027_str = etree.tostring(field_027_xml, pretty_print=True, encoding="unicode")

    print(field_027_str)
    return(field_027_xml)

In [84]:
def ead2marc_028(unitid_raw):
    '''INDICATORS'''
    '''Indicator 1'''
    unitid_type = unitid_raw.get("type", "").lower()
    field_2_028 = ""
    if "issue" in unitid_type:
        ind1_028 = "0"
    elif "matrix" in unitid_type:
        ind1_028 = "1"
    elif "plate" in unitid_type:
        ind1_028 = "2"
    elif "music" in unitid_type:
        ind1_028 = "3"
    elif "video" in unitid_type:
        ind1_028 = "4"
    elif "distributor" in unitid_type:
        ind1_028 = "6"
    else:
        ind1_028 = "5"
    '''Indicator 2 is constant (0)'''
    #NOTE: Only ind2 0 is currently supported
    
    '''SUBFIELDS'''
    '''SUBFIELD A'''
    unitid_str = unitid_raw.xpath("string()").strip()
    a_028 = f"""<subfield code="a">{unitid_str}</subfield>"""

    field_028_open = f"""<datafield tag="028" ind1="{ind1_028}" ind2=" ">"""
    field_028_str_nb = field_028_open + a_028 + field_2_028 + "</datafield>"
    field_028_xml = etree.fromstring(field_028_str_nb)
    field_028_str = etree.tostring(field_028_xml, pretty_print=True, encoding="unicode")

    print(field_028_str)
    return(field_028_xml)

In [85]:
def ead2marc_050(unitid_raw):
    unitid_str = unitid_raw.xpath("string()").strip()
    '''INDICATORS'''
    '''Indicator 1 is constant (blank)'''
    '''Indicator 2 is constant (4)'''

    '''SUBFIELDS'''
    #Checks for number of cutters and constrcuts $a content and $b content accordingly
    cn = pycn.callnumber(unitid_str)
    class_str = str(cn.classification)
    class_ns = class_str.replace(" ", "")
    if cn.edition:
        all_eds = str(cn.edition)
    else:
        all_eds = ""
    if cn.item:
        all_items = str(cn.item)
    else:
        all_items = ""
    first_cutter = "." + str(cn.cutters[0])
    if len(cn.cutters) <= 1:
        #Subfield content construction for call numbers with 1 cutter
        a_content = class_ns
        b_content = first_cutter + all_eds + all_items
    else:
        #Subfield content construction for call numbers with 2+ cutters
        second_on_cutters_raw = cn.cutters[1:]
        second_on_cutters_list = []
        for cutter in second_on_cutters_raw:
            cutter_str = str(cutter)
            second_on_cutters_list.append(cutter_str)
        second_on_cutters = " ".join(second_on_cutters_list)
        a_content = class_ns + first_cutter
        b_content = second_on_cutters + all_eds + all_items
        

    '''SUBFIELD A'''
    a_050 = f"""<subfield code="a">{a_content}</subfield>"""

    '''SUBFIELD B'''
    b_050 = f"""<subfield code="b">{b_content}</subfield>"""


    '''PRINT 050 FIELD'''
    field_050_open = f"""<datafield tag="050" ind1=" " ind2="4">"""
    field_050_str_nb = field_050_open + a_050 + b_050 + "</datafield>"
    field_050_xml = etree.fromstring(field_050_str_nb)
    field_050_str = etree.tostring(field_050_xml, pretty_print=True, encoding="unicode")

    print(field_050_str)
    return(field_050_xml)


In [86]:
def ead2marc_082(unitid_raw):
    unitid_str = unitid_raw.xpath("string()").strip()
    cn = pycn.callnumber(unitid_str)

    
    '''INDICATORS'''
    '''Indicator 1 is constant (0)'''
    '''Indicator 2 is constant (4)'''

    '''SUBFIELDS'''
    '''SUBFIELD A'''
    class_str = str(cn.classification)
    class_ns = class_str.replace(" ", "")
    a_082 = f"""<subfield code="a">{class_ns}</subfield>"""


    '''SUBFIELD B'''
    if cn.cutters:
        all_cutters = str(cn.cutters)
    else:
        all_cutters = ""
    if cn.item:
        all_items = str(cn.item)
    else:
        all_items = ""
    b_content = all_cutters + all_items
    b_082 = f"""<subfield code="b">{b_content}</subfield>"""

    '''PRINT 082 FIELD'''
    field_082_open = f"""<datafield tag="082" ind1="0" ind2="4">"""
    field_082_str_nb = field_082_open + a_082 + b_082 + "</datafield>"
    field_082_xml = etree.fromstring(field_082_str_nb)
    field_082_str = etree.tostring(field_082_xml, pretty_print=True, encoding="unicode")

    print(field_082_str)
    return(field_082_xml)

In [87]:
def ead2marc_086(unitid_raw):
    unitid_str = unitid_raw.xpath("string()").strip()
    '''INDICATORS'''
    '''Indicator 1 is constant (0)'''
    '''Indicator 2 is constant (blank)'''
    
    '''SUBFIELDS'''
    '''SUBFIELD A'''
    a_086 = f"""<subfield code="a">{unitid_str}</subfield>"""

    '''PRINT 086 FIELD'''
    field_086_open = f"""<datafield tag="086" ind1=" " ind2="4">"""
    field_086_str_nb = field_086_open + a_086 + "</datafield>"
    field_086_xml = etree.fromstring(field_086_str_nb)
    field_086_str = etree.tostring(field_086_xml, pretty_print=True, encoding="unicode")

    print(field_086_str)
    return(field_086_xml)

In [88]:
def ead2marc_02x_05x_08x(raw):
    unitid_list = raw.xpath(".//*[local-name()='unitid']")
    for unitid in unitid_list:
        unitid_type = unitid.get("type", "").lower()
        unitid_str = unitid.xpath("string()").strip()
        if unitid.get("localtype") == "aspace_uri":
            continue
        else:
            # (Troubleshot with Claude Opus 4.6)
            callnotest = pycn.callnumber(unitid_str)
            callno_type = type(callnotest).__name__
            if callno_type == "LC":
                ead2marc_050(unitid)
            elif callno_type == "Dewey":
                print("082")
            elif callno_type == "SuDoc":
                print("086")
            else:
                if (unitid_type == "isbn") or ("international standard book" in unitid_type):
                    ead2marc_020(unitid)
                elif unitid_type == "issn" or ("international standard serial" in unitid_type and not "cluster" in unitid_type):
                    ead2marc_022(unitid)
                elif (unitid_type == "issn-l" or unitid_type == "issn-h") or ("international standard serial" in unitid_type and "cluster" in unitid_type):
                    ead2marc_023(unitid)
                elif (unitid_type == "isrc") or ("international standard recording code" in unitid_type):
                    ead2marc_024(unitid)
                elif (unitid_type == "upc") or ("universal product code" in unitid_type):
                    ead2marc_024(unitid)
                elif (unitid_type == "ismn") or ("international standard music" in unitid_type):
                    ead2marc_024(unitid)
                elif (unitid_type == "ean") or ("international article number" in unitid_type):
                    ead2marc_024(unitid)
                elif (unitid_type == "sici") or ("serial item and contribution" in unitid_type):
                    ead2marc_024(unitid)
                elif unitid_type == "fingerprint" or unitid_type == "fingerprint identifier":
                    ead2marc_026(unitid)
                elif (unitid_type == "strn" or unitid_type == "isrn") or ("standard technical report" in unitid_type):
                    ead2marc_027(unitid)
                elif ("publisher" in unitid_type) or ("issue" in unitid_type) or ("matrix" in unitid_type) or ("plate" in unitid_type) or ("distributor" in unitid_type):
                    ead2marc_028(unitid)
                else:
                    ead2marc_024(unitid)

ead2marc_02x_05x_08x(c0_raw)

<datafield tag="028" ind1="5" ind2=" ">
  <subfield code="a">IMI
                7717</subfield>
</datafield>



In [89]:
def ead2marc_040():
    '''Creates 040 for IU Libraries (constant)'''
    field_040_open = """<datafield tag="040" ind1=" " ind2=" ">""" 
    a_040 = """<subfield code="a">IUL</subfield>""" 
    b_040 = """<subfield code="b">eng</subfield>""" 
    e_040 = """<subfield code="e">rda</subfield>""" 
    c_040 = """<subfield code="c">IUL</subfield>"""
    field_040_close = """</datafield>"""

    field_040_str_nb = field_040_open + a_040 + b_040 + e_040 + c_040 + field_040_close
    field_040_xml = etree.fromstring(field_040_str_nb)
    field_040_str = etree.tostring(field_040_xml, pretty_print=True, encoding="unicode")

    print(field_040_str)
    return(field_040_xml)

ead2marc_040()

<datafield tag="040" ind1=" " ind2=" ">
  <subfield code="a">IUL</subfield>
  <subfield code="b">eng</subfield>
  <subfield code="e">rda</subfield>
  <subfield code="c">IUL</subfield>
</datafield>



<Element datafield at 0x197291da180>

In [90]:
def ead2marc_041(raw):
    languageset_list = raw.xpath(".//*[local-name()='languageset']")
    a_041_list = []

    '''INDICATORS'''
    '''Indicator 1 is constant (blank)'''
    '''Indicator 2 is constant (7)'''

    '''SUBFIELDS'''
    '''SUBFIELD A'''
    for languageset in languageset_list:
        language_list = languageset.xpath(".//*[local-name()='language']")
        language = language_list[0]
        langcode = language.attrib["langcode"].lower()
        a_041 = f"""<subfield code="a">{langcode}</subfield>"""
        a_041_list.append(a_041)
    field_041_open = f"""<datafield tag="041" ind1=" " ind2="7">"""
    sf_2_041 = """<subfield code="2">iso639-2b</subfield>"""
    # Adds subfield 2 because EAD uses ISO639-2b, not MARC language codes
    field_041_str_nb = field_041_open + "".join(a_041_list) + sf_2_041 + "</datafield>"
    field_041_xml = etree.fromstring(field_041_str_nb)
    field_041_str = etree.tostring(field_041_xml, pretty_print=True, encoding="unicode")


    print(field_041_str)
    return(field_041_xml)

# TODO: Add support for indicators
# TODO: Add support for other subfields beyond $a
# TODO: Look into MARC to ISO639-2b mapping and implement

ead2marc_041(c0_raw)

<datafield tag="041" ind1=" " ind2="7">
  <subfield code="a">eng</subfield>
  <subfield code="2">iso639-2b</subfield>
</datafield>



<Element datafield at 0x1972640a980>

In [91]:
def ead2marc_049():
    '''Creates 049 for IU Libraries holdings (constant)'''
    field_049_open = """<datafield tag="049" ind1=" " ind2=" ">"""
    a_049 = """<subfield code="a">IULA</subfield>"""
    field_049_close = """</datafield>"""

    field_049_str_nb = field_049_open + a_049 + field_049_close
    field_049_xml = etree.fromstring(field_049_str_nb)
    field_049_str = etree.tostring(field_049_xml, pretty_print=True, encoding="unicode")

    print(field_049_str)
    return(field_049_xml)

ead2marc_049()

<datafield tag="049" ind1=" " ind2=" ">
  <subfield code="a">IULA</subfield>
</datafield>



<Element datafield at 0x19729204780>

In [92]:
def ead2marc_100(name):
    '''Check if main name is associated with an authority file'''
    # (This portion of code partially revised utilizing ChatGPT 5.2)
    '''Pull identifier'''
    if name.get("source") in {"lcnaf", "naf", "viaf"} and name.get("identifier"):
        authfile_no = name.get("identifier")
    elif name.get("source") in {"lcnaf", "naf"}:
        name_str = name.xpath("string()").strip()
        suggest_url = f"""https://id.loc.gov/authorities/names/suggest2?q={name_str}"""
        # (This portion of code was generated utilizing Claude Opus 4.5)
        suggest_response = requests.get(suggest_url)
        suggest_data = suggest_response.json()
        authfile_no = None
        for hit in suggest_data.get("hits", []):
            if hit["aLabel"] == name_str:
                authfile_no = hit["token"]
                break
    elif name.get("source") == "viaf":
         # (This portion of code was generated utilizing Claude Opus 4.5)
        name_str = name.xpath("string()").strip()
        viaf_search_url = f"""https://viaf.org/viaf/search?query=local.personalNames+all+%22{name_str}%22&sortKeys=holdingscount&maximumRecords=5"""
        viaf_headers = {'Accept': 'application/xml'}
        viaf_search_response = requests.get(viaf_search_url, headers=viaf_headers)
        viaf_search_root = etree.fromstring(viaf_search_response.content)
        authfile_no = None
        records = viaf_search_root.xpath('//*[local-name()="record"]')
        for rec in records:
            headings = rec.xpath('.//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_id = rec.xpath('.//*[local-name()="viafID"]')
            if headings and viaf_id and headings[0].text == name_str:
                authfile_no = viaf_id[0].text
                break
    else:
        authfile_no = None
        
    '''Pull authority file using identifier'''
    if name.get("source") in {"lcnaf", "naf"} and authfile_no:
        '''Get Library of Congress Name Authority MARC/XML and clean'''
        authority = "lcnaf"
        authority_url = f"""https://lccn.loc.gov/{authfile_no}/marcxml"""
        # print(authority_url)
        # print(requests.get(authority_url).status_code)
        authority_xml = requests.get(authority_url).content
        authority_root = etree.fromstring(authority_xml)
        authority_100_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='100']")
        authority_100_raw = authority_100_list[0]
        '''Clean authority_100_raw'''
        authority_100_str = etree.tostring(authority_100_raw, pretty_print=True, encoding="unicode")
        authority_100_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_100_str)
        authority_100_str_list = authority_100_str.split("\n")
        authority_100_str_list_stripped = [str.strip() for str in authority_100_str_list]
        authority_100_str = "".join(authority_100_str_list_stripped)
        authority_100_str = re.sub(r'</datafield>', '', authority_100_str).strip()  
    elif name.get("source") == "viaf" and authfile_no:
        '''Get VIAF cluster XML and extract 100 field data'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority = "viaf"
        viaf_headers = {'Accept': 'application/xml'}
        viaf_url = f"https://viaf.org/viaf/{authfile_no}"
        viaf_response = requests.get(viaf_url, headers=viaf_headers)
        viaf_root = etree.fromstring(viaf_response.content)
        '''Check if VIAF cluster has linked LCNAF -- if so, use LCNAF'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        lc_sources = viaf_root.xpath('//*[local-name()="source" and starts-with(text(), "LC|")]')
        if lc_sources:
            '''VIAF has LC link -- fetch from LCNAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            lc_id = lc_sources[0].text.split('|')[1]
            authority_url = f"https://lccn.loc.gov/{lc_id}/marcxml"
            authority_xml = requests.get(authority_url).content
            authority_root = etree.fromstring(authority_xml)
            authority_100_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='100']")
            authority_100_raw = authority_100_list[0]
            authority_100_str = etree.tostring(authority_100_raw, pretty_print=True, encoding="unicode")
            authority_100_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_100_str)
            authority_100_str_list = authority_100_str.split("\n")
            authority_100_str_list_stripped = [str.strip() for str in authority_100_str_list]
            authority_100_str = "".join(authority_100_str_list_stripped)
            authority_100_str = re.sub(r'</datafield>', '', authority_100_str).strip()
        else:
            '''No LC source -- parse VIAF cluster directly'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_headings = viaf_root.xpath('//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_main_heading = viaf_headings[0].text if viaf_headings else None
            '''Get normalized dates from VIAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_birth = viaf_root.xpath('//*[local-name()="birthDate"]')
            viaf_death = viaf_root.xpath('//*[local-name()="deathDate"]')
            viaf_birth_year = viaf_birth[0].text[:4] if viaf_birth and viaf_birth[0].text and not viaf_birth[0].text.startswith('0') else None
            viaf_death_year = viaf_death[0].text[:4] if viaf_death and viaf_death[0].text and not viaf_death[0].text.startswith('0') else None
            '''Parse heading to separate name from dates'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_parts = viaf_main_heading.split(', ') if viaf_main_heading else []
            viaf_name_parts = []
            for part in viaf_parts:
                if not (any(c.isdigit() for c in part) and ('-' in part or part.endswith('-'))):
                    viaf_name_parts.append(part)
            viaf_ind1 = '1' if len(viaf_name_parts) > 1 else '0'
            viaf_a_content = ', '.join(viaf_name_parts)
            '''Determine date subfield'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            if viaf_birth_year and viaf_death_year:
                viaf_d_content = f'{viaf_birth_year}-{viaf_death_year}'
            elif viaf_birth_year:
                viaf_d_content = f'{viaf_birth_year}-'
            else:
                viaf_d_content = None
            '''Build authority_100_str for VIAF-direct'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_subfields = f'<subfield code="a">{viaf_a_content}</subfield>'
            if viaf_d_content:
                viaf_subfields += f'<subfield code="d">{viaf_d_content}</subfield>'
            authority_100_str = f'<datafield tag="100" ind1="{viaf_ind1}" ind2=" ">{viaf_subfields}'
    else:
        authority = None

        
    '''INDICATORS'''
    '''Indicator 1'''
    if authority not in ["lcnaf", "viaf"]:
        a_content = name.xpath("string()").strip()
        a_alpha = []
        d_num = []
        a_split = a_content.split(", ")
        if name in creator_famnames_list:
            ind1_100 = "3"
        else:
            for item in a_split:
                if any(char.isnumeric() for char in item):
                    d_num.append(item)
                else:
                    a_alpha.append(item)
            if len(a_alpha) > 1:
                ind1_100 = "1"
            else:
                ind1_100 = "0"
    else:
        ind1_100 = ""

    '''Indicator 2 is constant (blank)'''

    
    '''SUBFIELD A'''
    if authority not in ["lcnaf", "viaf"]:
        if name in creator_famnames_list:
            a_content = a_content
        else:
            a_content = ", ".join(a_alpha)
        a_100 = f"""<subfield code ="a">{a_content}</subfield>"""

    '''SUBFIELD D'''
    if authority not in ["lcnaf", "viaf"] and name not in creator_famnames_list:
        d_content = d_num[0]
        d_content = d_content.rstrip(".")
        d_100 = f"""<subfield code ="d">{d_content}</subfield>"""
    else:
        d_100 = ""
    
    '''SUBFIELD E'''
    if 'relator' in name.attrib:
        aspace_relator = name.attrib["relator"].lower()
        if aspace_relator in marc_rda_relators.keys():
            e_content = marc_rda_relators[aspace_relator]
            e_100 = f"""<subfield code ="e">{e_content}</subfield>"""
        else:
            e_100 = ""
    else:
        e_100 = ""

    '''PRINT 100 FIELD'''
    if authority == "lcnaf" or authority == "viaf":
        field_100_str_nb = authority_100_str + e_100 + "</datafield>"
        field_100_xml = etree.fromstring(field_100_str_nb)
        field_100_str = etree.tostring(field_100_xml, pretty_print=True, encoding="unicode")
    else:
        field_100_open = f"""<datafield tag="100" ind1="{ind1_100}" ind2=" ">"""
        field_100_str_nb = field_100_open + a_100 + d_100 + e_100 + "</datafield>"
        field_100_xml = etree.fromstring(field_100_str_nb)
        field_100_str = etree.tostring(field_100_xml, pretty_print=True, encoding="unicode")
    
    '''Reorder attributes to put tag first'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    field_100_str = re.sub(r'<datafield ind1="(.)" ind2="(.)" tag="(\d+)">', r'<datafield tag="\3" ind1="\1" ind2="\2">', field_100_str)
    
    print(field_100_str)
    return field_100_xml
    
    # NOTE: 
        # Subfield C is not currently supported for non-lcaf/viaf name authorities.
        # Family names are not currently parsed and separated into subfields. All information placed in subfield A.

In [93]:
def ead2marc_110(name):
    '''Check if main name is associated with an authority file'''
    # (This portion of code partially revised utilizing ChatGPT 5.2)
    '''Pull identifier'''
    if name.get("source") in {"lcnaf", "naf", "viaf"} and name.get("identifier"):
        authfile_no = name.get("identifier")
    elif name.get("source") in {"lcnaf", "naf"}:
        name_str = name.xpath("string()").strip()
        suggest_url = f"""https://id.loc.gov/authorities/names/suggest2?q={name_str}"""
        # (This portion of code was generated utilizing Claude Opus 4.5)
        suggest_response = requests.get(suggest_url)
        suggest_data = suggest_response.json()
        authfile_no = None
        for hit in suggest_data.get("hits", []):
            if hit["aLabel"] == name_str:
                authfile_no = hit["token"]
                break
    elif name.get("source") == "viaf":
        # (This portion of code was generated utilizing Claude Opus 4.5)
        name_str = name.xpath("string()").strip()
        viaf_search_url = f"""https://viaf.org/viaf/search?query=local.corporateNames+all+%22{name_str}%22&sortKeys=holdingscount&maximumRecords=5"""
        viaf_headers = {'Accept': 'application/xml'}
        viaf_search_response = requests.get(viaf_search_url, headers=viaf_headers)
        viaf_search_root = etree.fromstring(viaf_search_response.content)
        authfile_no = None
        records = viaf_search_root.xpath('//*[local-name()="record"]')
        for rec in records:
            headings = rec.xpath('.//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_id = rec.xpath('.//*[local-name()="viafID"]')
            if headings and viaf_id and headings[0].text == name_str:
                authfile_no = viaf_id[0].text
                break
    else:
        authfile_no = None

    '''Pull authority file using identifier'''
    if name.get("source") in {"lcnaf", "naf"} and authfile_no:
        '''Get Library of Congress Name Authority MARC/XML and clean'''
        authority = "lcnaf"
        authority_url = f"""https://lccn.loc.gov/{authfile_no}/marcxml"""
        authority_xml = requests.get(authority_url).content
        authority_root = etree.fromstring(authority_xml)
        authority_110_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
        authority_110_raw = authority_110_list[0]
        '''Clean authority_110_raw'''
        authority_110_str = etree.tostring(authority_110_raw, pretty_print=True, encoding="unicode")
        authority_110_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_110_str)
        authority_110_str_list = authority_110_str.split("\n")
        authority_110_str_list_stripped = [str.strip() for str in authority_110_str_list]
        authority_110_str = "".join(authority_110_str_list_stripped)
        authority_110_str = re.sub(r'</datafield>', '', authority_110_str).strip()  
    elif name.get("source") == "viaf" and authfile_no:
        '''Get VIAF cluster XML and extract 110 field data'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority = "viaf"
        viaf_headers = {'Accept': 'application/xml'}
        viaf_url = f"https://viaf.org/viaf/{authfile_no}"
        viaf_response = requests.get(viaf_url, headers=viaf_headers)
        viaf_root = etree.fromstring(viaf_response.content)
        '''Check if VIAF cluster has linked LCNAF -- if so, use LCNAF'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        lc_sources = viaf_root.xpath('//*[local-name()="source" and starts-with(text(), "LC|")]')
        if lc_sources:
            '''VIAF has LC link -- fetch from LCNAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            lc_id = lc_sources[0].text.split('|')[1]
            authority_url = f"https://lccn.loc.gov/{lc_id}/marcxml"
            authority_xml = requests.get(authority_url).content
            authority_root = etree.fromstring(authority_xml)
            authority_110_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
            authority_110_raw = authority_110_list[0]
            authority_110_str = etree.tostring(authority_110_raw, pretty_print=True, encoding="unicode")
            authority_110_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_110_str)
            authority_110_str_list = authority_110_str.split("\n")
            authority_110_str_list_stripped = [str.strip() for str in authority_110_str_list]
            authority_110_str = "".join(authority_110_str_list_stripped)
            authority_110_str = re.sub(r'</datafield>', '', authority_110_str).strip()
        else:
            '''No LC source -- parse VIAF cluster directly'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_headings = viaf_root.xpath('//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_main_heading = viaf_headings[0].text if viaf_headings else None
            '''Build authority_110_str for VIAF-direct'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_subfields = f'<subfield code="a">{viaf_main_heading}</subfield>'
            authority_110_str = f'<datafield tag="110" ind1="2" ind2=" ">{viaf_subfields}'
    else:
        authority = None

    '''INDICATORS'''
    '''Indicator 1'''
    if authority not in ["lcnaf", "viaf"]:
        ind1_110 = "2"
    # TODO: Add support for ind1 0 (inverted name) and 1 (jurisdiction name)

    '''Indicator 2 is constant (blank)'''

    '''SUBFIELDS'''
    '''SUBFIELD A'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    if authority not in ["lcnaf", "viaf"]:
        a_content = name.xpath("string()").strip()
        a_110 = f"""<subfield code ="a">{a_content}</subfield>"""

    '''SUBFIELD E'''
    if 'relator' in name.attrib:
        aspace_relator = name.attrib["relator"].lower()
        if aspace_relator in marc_rda_relators.keys():
            e_content = marc_rda_relators[aspace_relator]
            e_110 = f"""<subfield code ="e">{e_content}</subfield>"""
        else:
            e_110 = ""
    else:
        e_110 = ""

    '''PRINT 110 FIELD'''
    if authority == "lcnaf" or authority == "viaf":
        field_110_str_nb = authority_110_str + e_110 + "</datafield>"
        field_110_xml = etree.fromstring(field_110_str_nb)
        field_110_str = etree.tostring(field_110_xml, pretty_print=True, encoding="unicode")
    else:
        field_110_open = f"""<datafield tag="110" ind1="{ind1_110}" ind2=" ">"""
        field_110_str_nb = field_110_open + a_110 + e_110 + "</datafield>"
        field_110_xml = etree.fromstring(field_110_str_nb)
        field_110_str = etree.tostring(field_110_xml, pretty_print=True, encoding="unicode")
    
    '''Reorder attributes to put tag first'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    field_110_str = re.sub(r'<datafield ind1="(.)" ind2="(.)" tag="(\d+)">', r'<datafield tag="\3" ind1="\1" ind2="\2">', field_110_str)
        
    print(field_110_str)
    return field_110_xml
    
    # NOTE:
        # Currently no subfields beyond subfields A and E are supported for non-authority 110s. All content except relators will be placed in subfield A.

In [94]:
def ead2marc_100_110(names_list):
    '''Set function-wide variables'''
    main_name_ead = names_list[0]
    
    '''Determine whether first name is persname or corpname to determine if field is 100 or 110'''
    '''Extract the actual name element (persname/famname/corpname) from the origination'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    if main_name_ead in creator_persnames_list:
        name_element = main_name_ead.xpath(".//*[local-name()='persname']")[0]
        field_100_xml = ead2marc_100(name_element)
        return field_100_xml
    elif main_name_ead in creator_famnames_list:
        name_element = main_name_ead.xpath(".//*[local-name()='famname']")[0]
        field_100_xml = ead2marc_100(name_element)
        return field_100_xml
    elif main_name_ead in creator_corpnames_list:
        name_element = main_name_ead.xpath(".//*[local-name()='corpname']")[0]
        field_110_xml = ead2marc_110(name_element)
        return field_110_xml
    # TODO: CREATE CASES FOR OTHER EAD NAME ELEMENTS (agencyname, geogname, namegrp)
    # TODO: CREATE SOME WAY FOR USER TO TOGGLE WHICH NAME THEY WANT TO BE 100/110 INSTEAD OF JUST SETTING IT TO FIRST LISTED CREATOR.
    
if creator_names_list:
    ead2marc_100_110(creator_names_list)

<datafield tag="100" ind1="1" ind2=" ">
  <subfield code="a">Adler, Ayal,</subfield>
  <subfield code="d">1968-</subfield>
  <subfield code="e">composer</subfield>
</datafield>



In [95]:
def ead2marc_245(raw):

    '''INDICATORS'''
    '''Indicator 1 is constant (1)'''
    '''Indicator 2 is constant (0)'''

    
    '''SUBFIELD A'''
    # (This portion of code was partially revised utilizing ChatGPT 5.2)
    title_fetch = raw.xpath(".//*[local-name()='unittitle']")
    title_raw = title_fetch[0]
    title_clean = title_raw.xpath("string()").strip()
    title_clean = html.escape(title_clean)
    title_clean = " ".join(title_clean.split())
    a_245 = f"""<subfield code="a">{title_clean}</subfield>"""
    
    '''PRINT 245 FIELD'''
    field_245_str_nb = """<datafield tag="245" ind1="1" ind2="0">""" + a_245 + "</datafield>"
    field_245_xml = etree.fromstring(field_245_str_nb)
    field_245_str = etree.tostring(field_245_xml, pretty_print=True, encoding="unicode")
    print(field_245_str)
    return field_245_xml

ead2marc_245(c0_raw)

<datafield tag="245" ind1="1" ind2="0">
  <subfield code="a">Crystallization, for symphony orchestra</subfield>
</datafield>



<Element datafield at 0x197268e93c0>

In [96]:
def ead2marc_264(raw):
    '''Create a list of all unitdates'''
    unitdates_list = raw.xpath(".//*[local-name()='unitdate']")
    field_264_xml_list = []
    
    if len(unitdates_list) > 0:
        '''Complete INDICATORS through PRINT 264 Field for each unitdate'''
        for unitdate in unitdates_list:

            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
        
            '''Indicator 2'''
            # (This portion of code was partially revised utilizing ChatGPT 5.2)
            '''Get date label (datechar)'''
            date_clean = unitdate.xpath("string()").strip()
            date_clean = html.escape(date_clean)
            datechar_clean = (unitdate.get("datechar")).strip()
            
            '''Set indicator 2 based on datechar'''
            if datechar_clean in ['broadcast', 'publication']:
                ind2_264 = 1 # Publication
            elif datechar_clean in ['issued']:
                ind2_264 = 3 # Manufacture
            elif datechar_clean in ['copyright']:
                ind2_264 = 4 # Copyright notice date
            else: 
                # AKA elif datechar_clean in ['creation', 'deaccession', 'digitized', 'event', 'existence', 'modified', 'other', 'record keeping', 'usage']
                ind2_264 = 0 # Production
        
            
            '''SUBFIELD C'''
            '''Set date to read [not after dddd] if datechar is "deaccession"'''
            # (Laikin's idea)
            if datechar_clean in ['deaccession']:
                qualifier_open = "[not after "
                qualifier_close = "]"
            else:
                qualifier_open = ""
                qualifier_close = ""
        
            c_264 = f"""<subfield code ="c">{qualifier_open}{date_clean}{qualifier_close}</subfield>"""
        
            
            '''PRINT 264 FIELD'''
            field_264_open = f"""<datafield tag="264" ind1=" " ind2="{ind2_264}">"""
            field_264_str_nb = field_264_open + c_264 + "</datafield>"
            field_264_xml = etree.fromstring(field_264_str_nb)
            field_264_str = etree.tostring(field_264_xml, pretty_print=True, encoding="unicode")
            print(field_264_str)
            field_264_xml_list.append(field_264_xml)

        return field_264_xml_list

ead2marc_264(c0_raw)

In [97]:
def ead2marc_300(raw):
    # (This portion of code was generated utilizing Claude Opus 4.6)
    '''Get physdescstructured and physdesc elements separately'''
    physdesc_structured_list = raw.xpath(".//*[local-name()='physdescstructured']")
    physdesc_list = raw.xpath(".//*[local-name()='physdesc']")
    field_300_xml_list = []
    consumed_physdescs = []
    
    if physdesc_list or physdesc_structured_list:
        '''If physdesc or phydescstructured elements exist, proceeds with processing'''
        '''Process each physdescstructured paired with its following physdesc sibling'''
        for pds in physdesc_structured_list:
            a_300_cs = ""
            a_300_qu = ""
            f_300 = ""
            c_300 = ""

            '''INDICATORS'''

            '''Indicator 1 is constant (blank)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''

            '''Process structured fields'''
            if pds.get("physdescstructuredtype") == "spaceoccupied":
                if pds.xpath(".//*[local-name()='quantity']"):
                    quantity_raw = pds.xpath(".//*[local-name()='quantity']")[0]
                    quantity_clean = html.escape(quantity_raw.xpath("string()").strip())
                else:
                    quantity_clean = ""
                if pds.xpath(".//*[local-name()='unittype']"):
                    unittype_raw = pds.xpath(".//*[local-name()='unittype']")[0]
                    unittype_clean = html.escape(unittype_raw.xpath("string()").strip())
                else:
                    unittype_clean = ""
                if quantity_clean and unittype_clean:
                    '''Subfield A (1/2)'''
                    a_300_qu = f"""<subfield code="a">{quantity_clean}</subfield>"""
                    '''Subfield F'''
                    f_300 = f"""<subfield code="f">{unittype_clean}</subfield>"""
                if pds.xpath(".//*[local-name()='dimensions']"):
                    dimensions_raw = pds.xpath(".//*[local-name()='dimensions']")[0]
                    dimensions_clean = html.escape(dimensions_raw.xpath("string()").strip())
                    '''Subfield C'''
                    c_300 = f"""<subfield code="c">{dimensions_clean}</subfield>"""
                # TODO: Move physfacets to ead2marc_500??

            '''Get paired following physdesc sibling (e.g. container_summary)'''
            following_physdesc = pds.xpath("following-sibling::*[local-name()='physdesc'][1]")
            if following_physdesc and following_physdesc[0].get("localtype") == "container_summary":
                fp = following_physdesc[0]
                fp_clean = html.escape(fp.xpath("string()").strip())
                '''Subfield A (2/2)'''
                a_300_cs = f"""<subfield code="a">{fp_clean}</subfield>"""
                consumed_physdescs.append(fp)

            '''Build 300 field'''
            field_300_open = """<datafield tag="300" ind1=" " ind2=" ">"""
            field_300_str_nb = field_300_open + a_300_cs + c_300 + a_300_qu + f_300 + "</datafield>"
            field_300_xml = etree.fromstring(field_300_str_nb)
            field_300_xml_list.append(field_300_xml)
            field_300_str = etree.tostring(field_300_xml, pretty_print=True, encoding="unicode")
            print(field_300_str)

    else:
        '''If no physdesc or phydescstructured elements exist, fallback is $a 1 [hierarchy level]'''
        field_300_open = """<datafield tag="300" ind1=" " ind2=" ">"""
        a_300 = c0_raw.attrib['level']
        field_300_str_nb = field_300_open + "1 " + a_300 + "</datafield>"
        field_300_xml = etree.fromstring(field_300_str_nb)
        field_300_xml_list.append(field_300_xml)
        field_300_str = etree.tostring(field_300_xml, pretty_print=True, encoding="unicode")
        print(field_300_str)

    return field_300_xml_list, consumed_physdescs
        # Consumed_physdescs included in return for use in ead2marc_500


ead2marc_300(c0_raw)

<datafield tag="300" ind1=" " ind2=" ">
  <subfield code="a">1 score (61 pages)</subfield>
  <subfield code="c">42 cm</subfield>
  <subfield code="a">1</subfield>
  <subfield code="f">folder(s)</subfield>
</datafield>



([<Element datafield at 0x19729277a40>],
 [<Element {http://ead3.archivists.org/schema/}physdesc at 0x19729277c40>])

In [98]:
def ead2marc_500(raw):
    field_300_result, consumed_physdescs = ead2marc_300(c0_raw)
    field_500_xml_list = []
    gnote_pref = {
        "odd": "",
        "dimensions": "Dimensions: ",
        "physdesc": "Physical Description note: ",
        "materialspec": "Material Specific Details: ",
        "physloc": "Location of resource: ",
        "phystech": "Physical Characteristics / Technical Requirements: ",
        "physfacet": "Physical Facet: ",
        "processinfo": "Processing Information: ",
        "separatedmaterial": "Materials Separated from the Resource: ",
    }
    gnote_list = raw.xpath(
        # (Troubleshot utilizing Claude Opus 4.6)
        ".//*[local-name()='odd' or local-name()='dimensions' or local-name()='physdesc' or local-name()='materialspec' or local-name()='physloc' or local-name()='phystech' or local-name()='physfacet' or local-name()='processinfo' or local-name()='separatedmaterial']"
    )
    if gnote_list:
        for gnote in gnote_list:
            if gnote in consumed_physdescs:
                # (This code generated utilizing Claude Opus 4.6)
                continue
            if etree.QName(gnote.getparent()).localname == 'physdescstructured':
                # (This code generated utilizing Claude Opus 4.6)
                continue
            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            gnote_type = etree.QName(gnote).localname
            if gnote_type in gnote_pref.keys():
                a_pref = gnote_pref[gnote_type]
            else:
                a_pref = ""
            gnote_clean = gnote.xpath("string()").strip(".")

            gnote_head_list = gnote.xpath(".//*[local-name()='head']")
            if gnote_head_list:
                gnote_head = gnote_head_list[0].xpath("string()").strip(".")
                gnote_clean = gnote_clean.replace(gnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            gnote_clean = " ".join(gnote_clean.split())
            gnote_clean = html.escape(gnote_clean)
            a_500 = f"""<subfield code ="a">{a_pref}{gnote_clean}</subfield>"""


            '''PRINT 500 FIELD'''
            field_500_str_nb = """<datafield tag="500" ind1=" " ind2=" ">""" + a_500 + "</datafield>"
            field_500_xml = etree.fromstring(field_500_str_nb)
            field_500_xml_list.append(field_500_xml)
            field_500_str = etree.tostring(field_500_xml, pretty_print=True, encoding="unicode")

            print(field_500_str)

        if field_500_xml_list:
            return field_500_xml_list


ead2marc_500(c0_raw)

<datafield tag="300" ind1=" " ind2=" ">
  <subfield code="a">1 score (61 pages)</subfield>
  <subfield code="c">42 cm</subfield>
  <subfield code="a">1</subfield>
  <subfield code="f">folder(s)</subfield>
</datafield>



In [99]:
def ead2marc_506(raw):
    field_506_xml_list = []
    ranote_list = raw.xpath(".//*[local-name()='accessrestrict']")
    if ranote_list:
        for ranote in ranote_list:
            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            # TODO: Add support for ind1 0 and 1
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            ranote_clean = ranote.xpath("string()").strip(".")
            ranote_head_list = ranote.xpath(".//*[local-name()='head']")
            if ranote_head_list:
                ranote_head = ranote_head_list[0].xpath("string()").strip(".")
                ranote_clean = ranote_clean.replace(ranote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            ranote_clean = " ".join(ranote_clean.split())
            ranote_clean = html.escape(ranote_clean)
            a_506 = f"""<subfield code ="a">{ranote_clean}</subfield>"""

            # TODO: Add support for other subfields

            '''PRINT 506 FIELD'''
            field_506_str_nb = """<datafield tag="506" ind1=" " ind2=" ">""" + a_506 + "</datafield>"
            field_506_xml = etree.fromstring(field_506_str_nb)
            field_506_xml_list.append(field_506_xml)
            field_506_str = etree.tostring(field_506_xml, pretty_print=True, encoding="unicode")

            print(field_506_str)

        if field_506_xml_list:
            return field_506_xml_list


ead2marc_506(c0_raw)

In [100]:
def ead2marc_520(raw):
    field_520_xml_list = []
    snote_ind1 = {
        "abstract": "3",
        "scopecontent": "2",
    }
    snote_list = raw.xpath(
        # (Troubleshot utilizing Claude Opus 4.6)
        ".//*[local-name()='abstract' or local-name()='scopecontent']"
    )
    if snote_list:
        for snote in snote_list:
            '''INDICATORS'''
            '''Indicator 1'''
            snote_type = etree.QName(snote).localname
            ind1_520 = snote_ind1[snote_type]
            field_520_open = f"""<datafield tag="520" ind1="{ind1_520}" ind2=" ">"""

            '''Indicator 2 is constant (blank)'''


            '''SUBFIELDS'''
            '''Subfield A'''
            snote_clean = snote.xpath("string()").strip(".")
            snote_head_list = snote.xpath(".//*[local-name()='head']")
            if snote_head_list:
                snote_head = snote_head_list[0].xpath("string()").strip(".")
                snote_clean = snote_clean.replace(snote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            snote_clean = " ".join(snote_clean.split())
            snote_clean = html.escape(snote_clean)
            a_520 = f"""<subfield code ="a">{snote_clean}</subfield>"""


            '''PRINT 520 FIELD'''
            field_520_str_nb = field_520_open + a_520 + "</datafield>"
            field_520_xml = etree.fromstring(field_520_str_nb)
            field_520_xml_list.append(field_520_xml)
            field_520_str = etree.tostring(field_520_xml, pretty_print=True, encoding="unicode")

            print(field_520_str)

        if field_520_xml_list:
            return field_520_xml_list


ead2marc_520(c0_raw)

<datafield tag="520" ind1="2" ind2=" ">
  <subfield code="a">Printed score for orchestra.</subfield>
</datafield>



[<Element datafield at 0x1972641e440>]

In [101]:
def ead2marc_524(raw):
    prefercite_list = raw.xpath(".//*[local-name()='prefercite']")
    if len(prefercite_list) > 0:
        field_524_xml_list = []
        for prefercite_raw in prefercite_list:

            '''INDICATORS'''
            '''Indicator 1 is constant (8)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''SUBFIELD A'''
            prefercite_clean = prefercite_raw.xpath("string()").strip(".")
            prefercite_head_list = prefercite_raw.xpath(".//*[local-name()='head']")
            if prefercite_head_list:
                prefercite_head = prefercite_head_list[0].xpath("string()").strip(".")
                prefercite_clean = prefercite_clean.replace(prefercite_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            prefercite_clean = " ".join(prefercite_clean.split())
            prefercite_clean = html.escape(prefercite_clean)
            a_524 = f"""<subfield code="a">{prefercite_clean}</subfield>"""

            '''PRINT 524 FIELD'''
            field_524_str_nb = """<datafield tag="524" ind1="8" ind2=" ">""" + a_524 + "</datafield>"
            field_524_xml = etree.fromstring(field_524_str_nb)
            field_524_xml_list.append(field_524_xml)
            field_524_str = etree.tostring(field_524_xml, pretty_print=True, encoding="unicode")

            print(field_524_str)

        return field_524_xml_list


ead2marc_524(c0_raw)

In [102]:
def ead2marc_535(raw):
    field_535_xml_list = []
    ldnote_ind1 = {
        "altformavail": "2",
        "originalsloc": "1",
    }
    ldnote_list = raw.xpath(
        ".//*[local-name()='altformavail' or local-name()='originalsloc']"
    )

    if ldnote_list:
        for ldnote in ldnote_list:
            '''INDICATORS'''
            '''Indicator 1'''
            ldnote_type = etree.QName(ldnote).localname
            ind1_535 = ldnote_ind1[ldnote_type]
            field_535_open = f"""<datafield tag="535" ind1="{ind1_535}" ind2=" ">"""

            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            ldnote_clean = ldnote.xpath("string()").strip(".")
            ldnote_head_list = ldnote.xpath(".//*[local-name()='head']")
            if ldnote_head_list:
                ldnote_head = ldnote_head_list[0].xpath("string()").strip(".")
                ldnote_clean = ldnote_clean.replace(ldnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            ldnote_clean = " ".join(ldnote_clean.split())
            ldnote_clean = html.escape(ldnote_clean)
            a_535 = f"""<subfield code ="a">{ldnote_clean}</subfield>"""


            '''PRINT 535 FIELD'''
            field_535_str_nb = field_535_open + a_535 + "</datafield>"
            field_535_xml = etree.fromstring(field_535_str_nb)
            field_535_xml_list.append(field_535_xml)
            field_535_str = etree.tostring(field_535_xml, pretty_print=True, encoding="unicode")

            print(field_535_str)

        if field_535_xml_list:
            return field_535_xml_list


ead2marc_535(c0_raw)

In [103]:
def ead2marc_540(raw):
    field_540_xml_list = []
    tgurnote_list = raw.xpath(
        ".//*[local-name()='userestrict' or local-name()='legalstatus']"
    )
    if tgurnote_list:
        for tgurnote in tgurnote_list:
            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            tgurnote_clean = tgurnote.xpath("string()").strip(".")
            tgurnote_head_list = tgurnote.xpath(".//*[local-name()='head']")
            if tgurnote_head_list:
                tgurnote_head = tgurnote_head_list[0].xpath("string()").strip(".")
                tgurnote_clean = tgurnote_clean.replace(tgurnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            tgurnote_clean = " ".join(tgurnote_clean.split())
            tgurnote_clean = html.escape(tgurnote_clean)
            a_540 = f"""<subfield code ="a">{tgurnote_clean}</subfield>"""

            '''PRINT 540 FIELD'''
            field_540_str_nb = """<datafield tag="540" ind1=" " ind2=" ">""" + a_540 + "</datafield>"
            field_540_xml = etree.fromstring(field_540_str_nb)
            field_540_xml_list.append(field_540_xml)
            field_540_str = etree.tostring(field_540_xml, pretty_print=True, encoding="unicode")

            print(field_540_str)

        if field_540_xml_list:
            return field_540_xml_list


ead2marc_540(c0_raw)

In [104]:
def ead2marc_541(raw):
    field_541_xml_list = []
    acqnote_list = raw.xpath(
        ".//*[local-name()='acqinfo']"
    )

    if acqnote_list:
        for acqnote in acqnote_list:
            '''INDICATORS'''
            '''Indicator 1'''
            # (Troubleshoot using Claude Opus 4.6)
            if acqnote.get("audience", "").lower() == "internal":
                ind1_541 = "0"
            else:
                ind1_541 = "1"
            field_541_open = f"""<datafield tag="541" ind1="{ind1_541}" ind2=" ">"""

            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            acqnote_clean = acqnote.xpath("string()").strip(".")
            acqnote_head_list = acqnote.xpath(".//*[local-name()='head']")
            if acqnote_head_list:
                acqnote_head = acqnote_head_list[0].xpath("string()").strip(".")
                acqnote_clean = acqnote_clean.replace(acqnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            acqnote_clean = " ".join(acqnote_clean.split())
            acqnote_clean = html.escape(acqnote_clean)
            a_541 = f"""<subfield code ="a">{acqnote_clean}</subfield>"""


            '''PRINT 541 FIELD'''
            field_541_str_nb = field_541_open + a_541 + "</datafield>"
            field_541_xml = etree.fromstring(field_541_str_nb)
            field_541_xml_list.append(field_541_xml)
            field_541_str = etree.tostring(field_541_xml, pretty_print=True, encoding="unicode")

            print(field_541_str)

        if field_541_xml_list:
            return field_541_xml_list


ead2marc_541(c0_raw)

In [105]:
def ead2marc_544(raw):
    field_544_xml_list = []
    loamnote_list = raw.xpath(
        ".//*[local-name()='relatedmaterial']"
    )
    if loamnote_list:
        for loamnote in loamnote_list:
            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            loamnote_clean = loamnote.xpath("string()").strip(".")
            loamnote_head_list = loamnote.xpath(".//*[local-name()='head']")
            if loamnote_head_list:
                loamnote_head = loamnote_head_list[0].xpath("string()").strip(".")
                loamnote_clean = loamnote_clean.replace(loamnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            loamnote_clean = " ".join(loamnote_clean.split())
            loamnote_clean = html.escape(loamnote_clean)
            a_544 = f"""<subfield code ="a">{loamnote_clean}</subfield>"""

            '''PRINT 544 FIELD'''
            field_544_str_nb = """<datafield tag="544" ind1=" " ind2=" ">""" + a_544 + "</datafield>"
            field_544_xml = etree.fromstring(field_544_str_nb)
            field_544_xml_list.append(field_544_xml)
            field_544_str = etree.tostring(field_544_xml, pretty_print=True, encoding="unicode")

            print(field_544_str)

        if field_544_xml_list:
            return field_544_xml_list


ead2marc_544(c0_raw)

In [106]:
def ead2marc_545(raw):
    field_545_xml_list = []
    bhnote_list = raw.xpath(
        ".//*[local-name()='bioghist']"
    )
    if bhnote_list:
        for bhnote in bhnote_list:
            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            bhnote_clean = bhnote.xpath("string()").strip(".")
            bhnote_head_list = bhnote.xpath(".//*[local-name()='head']")
            if bhnote_head_list:
                bhnote_head = bhnote_head_list[0].xpath("string()").strip(".")
                bhnote_clean = bhnote_clean.replace(bhnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            bhnote_clean = " ".join(bhnote_clean.split())
            bhnote_clean = html.escape(bhnote_clean)
            a_545 = f"""<subfield code ="a">{bhnote_clean}</subfield>"""

            '''PRINT 545 FIELD'''
            field_545_str_nb = """<datafield tag="545" ind1=" " ind2=" ">""" + a_545 + "</datafield>"
            field_545_xml = etree.fromstring(field_545_str_nb)
            field_545_xml_list.append(field_545_xml)
            field_545_str = etree.tostring(field_545_xml, pretty_print=True, encoding="unicode")

            print(field_545_str)

        if field_545_xml_list:
            return field_545_xml_list


ead2marc_545(c0_raw)

In [107]:
def ead2marc_546(raw):
    langmaterial_list = raw.xpath(".//*[local-name()='langmaterial']")
    languageset_list = raw.xpath(".//*[local-name()='languageset']")
    if len(langmaterial_list) > 0:
        field_546_xml_list = []
        for langmaterial_raw in langmaterial_list:

            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            
            langnote_fetch = langmaterial_raw.xpath(".//*[local-name()='descriptivenote']")
            if langnote_fetch:
                # Uses descriptive language note if one exists
                '''SUBFIELD A'''
                langnote_raw = langnote_fetch[0]
                langnote_clean = langnote_raw.xpath("string()").strip(".")
                langnote_head_list = langnote_raw.xpath(".//*[local-name()='head']")
                if langnote_head_list:
                    langnote_head = langnote_head_list[0].xpath("string()").strip(".")
                    langnote_clean = langnote_clean.replace(langnote_head, "")
                # (This portion of code was generated utilizing Claude Opus 4.6)
                langnote_clean = " ".join(langnote_clean.split())
                langnote_clean = html.escape(langnote_clean)
                a_546 = f"""<subfield code="a">{langnote_clean}</subfield>"""
            else:
                # If descriptive language note does not exist, $a is [Language 1], [Language 2], ...
                language_clean_list = []
                for languageset in languageset_list:
                    '''SUBFIELD A'''
                    language_fetch = languageset.xpath(".//*[local-name()='language']")
                    language_raw = language_fetch[0]
                    language_clean = language_raw.xpath("string()").strip(".")
                    language_clean = html.escape(language_clean)
                    language_clean_list.append(language_clean)
                    languages = (", ").join(language_clean_list)
                    a_546 = f"""<subfield code="a">{languages}</subfield>"""


            '''PRINT 546 FIELD'''
            field_546_str_nb = """<datafield tag="546" ind1=" " ind2=" ">""" + a_546 + "</datafield>"
            field_546_xml = etree.fromstring(field_546_str_nb)
            field_546_xml_list.append(field_546_xml)
            field_546_str = etree.tostring(field_546_xml, pretty_print=True, encoding="unicode")

            print(field_546_str)

        return field_546_xml_list


ead2marc_546(c0_raw)

<datafield tag="546" ind1=" " ind2=" ">
  <subfield code="a">English</subfield>
</datafield>



[<Element datafield at 0x197268e9a40>]

In [108]:
def ead2marc_555(raw):
    field_555_xml_list = []
    if vaid_clean:

        '''INDICATORS'''
        '''Indicator 1 is constant (blank)'''
        '''Indicator 2 is constant (blank)'''


        '''SUBFIELDS'''
        '''SUBFIELD A'''
        a_555 = """<subfield code="a">Finding aid (work): </subfield>"""

        '''SUBFIELD U'''
        faid_uri = f"""https://archives.iu.edu/catalog/{vaid_clean}"""
        u_555 = f"""<subfield code="u">{faid_uri} </subfield>"""

        
        '''PRINT 555 FIELD'''
        field_555_str_nb = """<datafield tag="555" ind1=" " ind2=" ">""" + a_555 + u_555 + "</datafield>"
        field_555_xml = etree.fromstring(field_555_str_nb)
        field_555_xml_list.append(field_555_xml)
        field_555_str = etree.tostring(field_555_xml, pretty_print=True, encoding="unicode")
        
        print(field_555_str)

    return field_555_xml_list


ead2marc_555(c0_raw)

<datafield tag="555" ind1=" " ind2=" ">
  <subfield code="a">Finding aid (work): </subfield>
  <subfield code="u">https://archives.iu.edu/catalog/VAE4896 </subfield>
</datafield>



[<Element datafield at 0x19726908a40>]

In [109]:
def ead2marc_561(raw):
    field_561_xml_list = []
    chnote_list = raw.xpath(
        ".//*[local-name()='custodhist']"
    )

    if chnote_list:
        for chnote in chnote_list:
            '''INDICATORS'''
            '''Indicator 1'''
            # (Troubleshoot using Claude Opus 4.6)
            if chnote.get("audience", "").lower() == "internal":
                ind1_561 = "0"
            else:
                ind1_561 = "1"
            field_561_open = f"""<datafield tag="561" ind1="{ind1_561}" ind2=" ">"""

            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            chnote_clean = chnote.xpath("string()").strip(".")
            chnote_head_list = chnote.xpath(".//*[local-name()='head']")
            if chnote_head_list:
                chnote_head = chnote_head_list[0].xpath("string()").strip(".")
                chnote_clean = chnote_clean.replace(chnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            chnote_clean = " ".join(chnote_clean.split())
            chnote_clean = html.escape(chnote_clean)
            a_561 = f"""<subfield code ="a">{chnote_clean}</subfield>"""


            '''PRINT 561 FIELD'''
            field_561_str_nb = field_561_open + a_561 + "</datafield>"
            field_561_xml = etree.fromstring(field_561_str_nb)
            field_561_xml_list.append(field_561_xml)
            field_561_str = etree.tostring(field_561_xml, pretty_print=True, encoding="unicode")

            print(field_561_str)

        if field_561_xml_list:
            return field_561_xml_list


ead2marc_561(c0_raw)

In [110]:
def ead2marc_583(raw):
    field_583_xml_list = []
    aprnote_list = raw.xpath(
        ".//*[local-name()='appraisal']"
    )

    if aprnote_list:
        for aprnote in aprnote_list:
            '''INDICATORS'''
            '''Indicator 1'''
            # (Troubleshoot using Claude Opus 4.6)
            if aprnote.get("audience", "").lower() == "internal":
                ind1_583 = "0"
            else:
                ind1_583 = "1"
            field_583_open = f"""<datafield tag="583" ind1="{ind1_583}" ind2=" ">"""

            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            aprnote_clean = aprnote.xpath("string()").strip(".")
            aprnote_head_list = aprnote.xpath(".//*[local-name()='head']")
            if aprnote_head_list:
                aprnote_head = aprnote_head_list[0].xpath("string()").strip(".")
                aprnote_clean = aprnote_clean.replace(aprnote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            aprnote_clean = " ".join(aprnote_clean.split())
            aprnote_clean = html.escape(aprnote_clean)
            a_583 = f"""<subfield code ="a">{aprnote_clean}</subfield>"""


            '''PRINT 583 FIELD'''
            field_583_str_nb = field_583_open + a_583 + "</datafield>"
            field_583_xml = etree.fromstring(field_583_str_nb)
            field_583_xml_list.append(field_583_xml)
            field_583_str = etree.tostring(field_583_xml, pretty_print=True, encoding="unicode")

            print(field_583_str)

        if field_583_xml_list:
            return field_583_xml_list


ead2marc_583(c0_raw)

In [111]:
def ead2marc_584(raw):
    field_584_xml_list = []
    afunote_list = raw.xpath(
        ".//*[local-name()='accruals']"
    )
    if afunote_list:
        for afunote in afunote_list:
            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2 is constant (blank)'''

            '''SUBFIELDS'''
            '''Subfield A'''
            afunote_clean = afunote.xpath("string()").strip(".")
            afunote_head_list = afunote.xpath(".//*[local-name()='head']")
            if afunote_head_list:
                afunote_head = afunote_head_list[0].xpath("string()").strip(".")
                afunote_clean = afunote_clean.replace(afunote_head, "")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            afunote_clean = " ".join(afunote_clean.split())
            afunote_clean = html.escape(afunote_clean)
            a_584 = f"""<subfield code ="a">{afunote_clean}</subfield>"""

            '''PRINT 584 FIELD'''
            field_584_str_nb = """<datafield tag="584" ind1=" " ind2=" ">""" + a_584 + "</datafield>"
            field_584_xml = etree.fromstring(field_584_str_nb)
            field_584_xml_list.append(field_584_xml)
            field_584_str = etree.tostring(field_584_xml, pretty_print=True, encoding="unicode")

            print(field_584_str)

        if field_584_xml_list:
            return field_584_xml_list


ead2marc_584(c0_raw)

In [112]:
def ead2marc_600(name):
    '''Check if name is associated with an authority file'''
    # (This portion of code partially revised utilizing ChatGPT 5.2)
    '''Pull identifier'''
    if name.get("source") in {"lcnaf", "naf", "viaf"} and name.get("identifier"):
        authfile_no = name.get("identifier")
    elif name.get("source") in {"lcnaf", "naf"}:
        name_str = name.xpath("string()").strip()
        suggest_url = f"""https://id.loc.gov/authorities/names/suggest2?q={name_str}"""
        # (This portion of code was generated utilizing Claude Opus 4.5)
        suggest_response = requests.get(suggest_url)
        suggest_data = suggest_response.json()
        authfile_no = None
        for hit in suggest_data.get("hits", []):
            if hit["aLabel"] == name_str:
                authfile_no = hit["token"]
                break
    elif name.get("source") == "viaf":
        # (This portion of code was generated utilizing Claude Opus 4.5)
        name_str = name.xpath("string()").strip()
        viaf_search_url = f"""https://viaf.org/viaf/search?query=local.personalNames+all+%22{name_str}%22&sortKeys=holdingscount&maximumRecords=5"""
        viaf_headers = {'Accept': 'application/xml'}
        viaf_search_response = requests.get(viaf_search_url, headers=viaf_headers)
        viaf_search_root = etree.fromstring(viaf_search_response.content)
        authfile_no = None
        records = viaf_search_root.xpath('//*[local-name()="record"]')
        for rec in records:
            headings = rec.xpath('.//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_id = rec.xpath('.//*[local-name()="viafID"]')
            if headings and viaf_id and headings[0].text == name_str:
                authfile_no = viaf_id[0].text
                break
    else:
        authfile_no = None

    '''Pull authority file using identifier'''
    if name.get("source") in {"lcnaf", "naf"} and authfile_no:
        '''Get Library of Congress Name Authority MARC/XML and clean'''
        authority = "lcnaf"
        authfile_no = name.get("identifier")
        authority_url = f"""https://lccn.loc.gov/{authfile_no}/marcxml"""
        authority_xml = requests.get(authority_url).content
        authority_root = etree.fromstring(authority_xml)
        authority_600_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='100']")
        # (This portion of code was generated utilizing Claude Opus 4.6)
        '''Fallback to tag 110 if tag 100 not found (handles mismatched EAD name types)'''
        if not authority_600_list:
            authority_600_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
        authority_600_raw = authority_600_list[0]
        '''Clean authority_600_raw'''
        authority_600_str = etree.tostring(authority_600_raw, pretty_print=True, encoding="unicode")
        authority_600_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_600_str)
        authority_600_str_list = authority_600_str.split("\n")
        authority_600_str_list_stripped = [str.strip() for str in authority_600_str_list]
        authority_600_str = "".join(authority_600_str_list_stripped)
        authority_600_str = re.sub(r'</datafield>', '', authority_600_str).strip()
        '''Change tag from 100 to 600'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority_600_str = authority_600_str.replace('tag="100"', 'tag="600"')
        # (This portion of code was generated utilizing Claude Opus 4.6)
        authority_600_str = authority_600_str.replace('tag="110"', 'tag="710"')
    elif name.get("source") == "viaf" and authfile_no:
        '''Get VIAF cluster XML and extract 600 field data'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority = "viaf"
        viaf_headers = {'Accept': 'application/xml'}
        viaf_url = f"https://viaf.org/viaf/{authfile_no}"
        viaf_response = requests.get(viaf_url, headers=viaf_headers)
        viaf_root = etree.fromstring(viaf_response.content)
        '''Check if VIAF cluster has linked LCNAF -- if so, use LCNAF'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        lc_sources = viaf_root.xpath('//*[local-name()="source" and starts-with(text(), "LC|")]')
        if lc_sources:
            '''VIAF has LC link -- fetch from LCNAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            lc_id = lc_sources[0].text.split('|')[1]
            authority_url = f"https://lccn.loc.gov/{lc_id}/marcxml"
            authority_xml = requests.get(authority_url).content
            authority_root = etree.fromstring(authority_xml)
            authority_600_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='100']")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            '''Fallback to tag 110 if tag 100 not found (handles mismatched EAD name types)'''
            if not authority_600_list:
                authority_600_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
            authority_600_raw = authority_600_list[0]
            authority_600_str = etree.tostring(authority_600_raw, pretty_print=True, encoding="unicode")
            authority_600_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_600_str)
            authority_600_str_list = authority_600_str.split("\n")
            authority_600_str_list_stripped = [str.strip() for str in authority_600_str_list]
            authority_600_str = "".join(authority_600_str_list_stripped)
            authority_600_str = re.sub(r'</datafield>', '', authority_600_str).strip()
            '''Change tag from 100 to 600'''
            authority_600_str = authority_600_str.replace('tag="100"', 'tag="600"')
            # (This portion of code was generated utilizing Claude Opus 4.6)
            authority_600_str = authority_600_str.replace('tag="110"', 'tag="710"')
        else:
            '''No LC source -- parse VIAF cluster directly'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_headings = viaf_root.xpath('//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_main_heading = viaf_headings[0].text if viaf_headings else None
            '''Get normalized dates from VIAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_birth = viaf_root.xpath('//*[local-name()="birthDate"]')
            viaf_death = viaf_root.xpath('//*[local-name()="deathDate"]')
            viaf_birth_year = viaf_birth[0].text[:4] if viaf_birth and viaf_birth[0].text and not viaf_birth[0].text.startswith('0') else None
            viaf_death_year = viaf_death[0].text[:4] if viaf_death and viaf_death[0].text and not viaf_death[0].text.startswith('0') else None
            '''Parse heading to separate name from dates'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_parts = viaf_main_heading.split(', ') if viaf_main_heading else []
            viaf_name_parts = []
            for part in viaf_parts:
                if not (any(c.isdigit() for c in part) and ('-' in part or part.endswith('-'))):
                    viaf_name_parts.append(part)
            viaf_ind1 = '1' if len(viaf_name_parts) > 1 else '0'
            viaf_a_content = ', '.join(viaf_name_parts)
            '''Determine date subfield'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            if viaf_birth_year and viaf_death_year:
                viaf_d_content = f'{viaf_birth_year}-{viaf_death_year}'
            elif viaf_birth_year:
                viaf_d_content = f'{viaf_birth_year}-'
            else:
                viaf_d_content = None
            '''Build authority_600_str for VIAF-direct'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_subfields = f'<subfield code="a">{viaf_a_content}</subfield>'
            if viaf_d_content:
                viaf_subfields += f'<subfield code="d">{viaf_d_content}</subfield>'
            authority_600_str = f'<datafield tag="600" ind1="{viaf_ind1}" ind2=" ">{viaf_subfields}'
    else:
        authority = None

    '''INDICATORS'''
    '''Indicator 1'''
    if authority not in ["lcnaf", "viaf"]:
        a_content = name.xpath("string()").strip()
        a_alpha = []
        d_num = []
        a_split = a_content.split(", ")
        for item in a_split:
            if any(char.isnumeric() for char in item):
                d_num.append(item)
            else:
                a_alpha.append(item)
        if len(a_alpha) > 1:
            ind1_600 = "1"
        else:
            ind1_600 = "0"
    else:
        ind1_600 = ""

    '''Indicator 2 is constant (blank)'''

    '''SUBFIELD A'''
    if authority not in ["lcnaf", "viaf"]:
        a_content = ", ".join(a_alpha)
        a_600 = f"""<subfield code ="a">{a_content}</subfield>"""

    '''SUBFIELD D'''
    if authority not in ["lcnaf", "viaf"] and d_num:
        d_content = d_num[0]
        d_content = d_content.rstrip(".")
        d_600 = f"""<subfield code ="d">{d_content}</subfield>"""
    else:
        d_600 = ""

    '''SUBFIELD E'''
    if 'relator' in name.attrib:
        aspace_relator = name.attrib["relator"].lower()
        if aspace_relator in marc_rda_relators.keys():
            e_content = marc_rda_relators[aspace_relator]
            e_600 = f"""<subfield code ="e">{e_content}</subfield>"""
        else:
            e_600 = ""
    else:
        e_600 = ""

    '''PRINT 600 FIELD'''
    if authority == "lcnaf" or authority == "viaf":
        field_600_str_nb = authority_600_str + e_600 + "</datafield>"
        field_600_xml = etree.fromstring(field_600_str_nb)
        field_600_str = etree.tostring(field_600_xml, pretty_print=True, encoding="unicode")
    else:
        field_600_open = f"""<datafield tag="600" ind1="{ind1_600}" ind2=" ">"""
        field_600_str_nb = field_600_open + a_600 + d_600 + e_600 + "</datafield>"
        field_600_xml = etree.fromstring(field_600_str_nb)
        field_600_str = etree.tostring(field_600_xml, pretty_print=True, encoding="unicode")

    '''Reorder attributes to put tag first'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    field_600_str = re.sub(r'<datafield ind1="(.)" ind2="(.)" tag="(\d+)">', r'<datafield tag="\3" ind1="\1" ind2="\2">', field_600_str)

    print(field_600_str)
    # (This portion of code was generated utilizing Claude Opus 4.6)
    return field_600_xml

In [113]:
def ead2marc_610(name):
    '''Check if main name is associated with an authority file'''
    # (This portion of code partially revised utilizing ChatGPT 5.2)
    '''Pull identifier'''
    if name.get("source") in {"lcnaf", "naf", "viaf"} and name.get("identifier"):
        authfile_no = name.get("identifier")
    elif name.get("source") in {"lcnaf", "naf"}:
        name_str = name.xpath("string()").strip()
        suggest_url = f"""https://id.loc.gov/authorities/names/suggest2?q={name_str}"""
        # (This portion of code was generated utilizing Claude Opus 4.5)
        suggest_response = requests.get(suggest_url)
        suggest_data = suggest_response.json()
        authfile_no = None
        for hit in suggest_data.get("hits", []):
            if hit["aLabel"] == name_str:
                authfile_no = hit["token"]
                break
    elif name.get("source") == "viaf":
        # (This portion of code was generated utilizing Claude Opus 4.5)
        name_str = name.xpath("string()").strip()
        viaf_search_url = f"""https://viaf.org/viaf/search?query=local.corporateNames+all+%22{name_str}%22&sortKeys=holdingscount&maximumRecords=5"""
        viaf_headers = {'Accept': 'application/xml'}
        viaf_search_response = requests.get(viaf_search_url, headers=viaf_headers)
        viaf_search_root = etree.fromstring(viaf_search_response.content)
        authfile_no = None
        records = viaf_search_root.xpath('//*[local-name()="record"]')
        for rec in records:
            headings = rec.xpath('.//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_id = rec.xpath('.//*[local-name()="viafID"]')
            if headings and viaf_id and headings[0].text == name_str:
                authfile_no = viaf_id[0].text
                break
    else:
        authfile_no = None

    '''Pull authority file using identifier'''
    if name.get("source") in {"lcnaf", "naf"} and authfile_no:
        '''Get Library of Congress Name Authority MARC/XML and clean'''
        authority = "lcnaf"
        authority_url = f"""https://lccn.loc.gov/{authfile_no}/marcxml"""
        authority_xml = requests.get(authority_url).content
        authority_root = etree.fromstring(authority_xml)
        authority_610_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
        authority_610_raw = authority_610_list[0]
        '''Clean authority_610_raw'''
        authority_610_str = etree.tostring(authority_610_raw, pretty_print=True, encoding="unicode")
        authority_610_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_610_str)
        authority_610_str_list = authority_610_str.split("\n")
        authority_610_str_list_stripped = [str.strip() for str in authority_610_str_list]
        authority_610_str = "".join(authority_610_str_list_stripped)
        authority_610_str = re.sub(r'</datafield>', '', authority_610_str).strip()
        '''Change tag from 110 to 610'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority_610_str = authority_610_str.replace('tag="110"', 'tag="610"')
    elif name.get("source") == "viaf" and authfile_no:
        '''Get VIAF cluster XML and extract 610 field data'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority = "viaf"
        viaf_headers = {'Accept': 'application/xml'}
        viaf_url = f"https://viaf.org/viaf/{authfile_no}"
        viaf_response = requests.get(viaf_url, headers=viaf_headers)
        viaf_root = etree.fromstring(viaf_response.content)
        '''Check if VIAF cluster has linked LCNAF -- if so, use LCNAF'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        lc_sources = viaf_root.xpath('//*[local-name()="source" and starts-with(text(), "LC|")]')
        if lc_sources:
            '''VIAF has LC link -- fetch from LCNAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            lc_id = lc_sources[0].text.split('|')[1]
            authority_url = f"https://lccn.loc.gov/{lc_id}/marcxml"
            authority_xml = requests.get(authority_url).content
            authority_root = etree.fromstring(authority_xml)
            authority_610_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
            authority_610_raw = authority_610_list[0]
            authority_610_str = etree.tostring(authority_610_raw, pretty_print=True, encoding="unicode")
            authority_610_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_610_str)
            authority_610_str_list = authority_610_str.split("\n")
            authority_610_str_list_stripped = [str.strip() for str in authority_610_str_list]
            authority_610_str = "".join(authority_610_str_list_stripped)
            authority_610_str = re.sub(r'</datafield>', '', authority_610_str).strip()
            '''Change tag from 110 to 610'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            authority_610_str = authority_610_str.replace('tag="110"', 'tag="610"')
        else:
            '''No LC source -- parse VIAF cluster directly'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_headings = viaf_root.xpath('//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_main_heading = viaf_headings[0].text if viaf_headings else None
            '''Build authority_610_str for VIAF-direct'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_subfields = f'<subfield code="a">{viaf_main_heading}</subfield>'
            authority_610_str = f'<datafield tag="610" ind1="2" ind2=" ">{viaf_subfields}'
    else:
        authority = None

    '''INDICATORS'''
    '''Indicator 1'''
    if authority not in ["lcnaf", "viaf"]:
        ind1_610 = "2"
    # TODO: Add support for ind1 0 (inverted name) and 1 (jurisdiction name)

    '''Indicator 2 is constant (blank)'''

    '''SUBFIELDS'''
    '''SUBFIELD A'''
    if authority not in ["lcnaf", "viaf"]:
        a_content = name.xpath("string()").strip()
        a_610 = f"""<subfield code ="a">{a_content}</subfield>"""

    '''SUBFIELD E'''
    if 'relator' in name.attrib:
        aspace_relator = name.attrib["relator"].lower()
        if aspace_relator in marc_rda_relators.keys():
            e_content = marc_rda_relators[aspace_relator]
            e_610 = f"""<subfield code ="e">{e_content}</subfield>"""
        else:
            e_610 = ""
    else:
        e_610 = ""

    '''PRINT 610 FIELD'''
    if authority == "lcnaf" or authority == "viaf":
        field_610_str_nb = authority_610_str + e_610 + "</datafield>"
        field_610_xml = etree.fromstring(field_610_str_nb)
        field_610_str = etree.tostring(field_610_xml, pretty_print=True, encoding="unicode")
    else:
        field_610_open = f"""<datafield tag="610" ind1="{ind1_610}" ind2=" ">"""
        field_610_str_nb = field_610_open + a_610 + e_610 + "</datafield>"
        field_610_xml = etree.fromstring(field_610_str_nb)
        field_610_str = etree.tostring(field_610_xml, pretty_print=True, encoding="unicode")
    
    '''Reorder attributes to put tag first'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    field_610_str = re.sub(r'<datafield ind1="(.)" ind2="(.)" tag="(\d+)">', r'<datafield tag="\3" ind1="\1" ind2="\2">', field_610_str)
        
    print(field_610_str)
    return field_610_xml
    
    # NOTE:
        # Currently no subfields beyond subfields A and E are supported for non-authority 610s. All content except relators will be placed in subfield A.

In [114]:
def ead2marc_630(title):
    authority_raw = title.get("source")
    if authority_raw == "lcac":
        authority = "cyac"
    else:
        authority = authority_raw
    title_clean = title.xpath("string()").strip()
    title_clean = html.escape(title_clean)

    '''INDICATORS'''
    '''Indicator 1 is constant (0)'''
    '''Indicator 2'''
    if authority == "lcsh":
        ind2_630 = "0"
    elif authority == "cyac":
        ind2_630 = "1"
    elif authority == "mesh":
        ind2_630 = "2"
    elif authority == "nal":
        ind2_630 = "3"
    elif authority == "":
        ind2_630 = " "
    elif authority == "cash":
        ind2_630 = "5"
    elif authority == "rvm":
        ind2_630 = "6"
    else:
        ind2_630 = "7"

    field_630_open = f"""<datafield tag="630" ind1="0" ind2="{ind2_630}">"""


    '''SUBFIELD A'''
    a_630 = f"""<subfield code="a">{title_clean}</subfield>"""

    '''SUBFIELD F'''
    if ind2_630 == "7":
        f_630 = f"""<subfield code="2">{authority}</subfield>"""
    else:
        f_630 = ""

    '''PRINT 630 FIELD'''
    # (This portion of code was troubleshot using Claud Opus 4.5)
    field_630_str_nb = field_630_open + a_630 + f_630 + "</datafield>"
    field_630_xml = etree.fromstring(field_630_str_nb)
    field_630_str = etree.tostring(field_630_xml, pretty_print=True, encoding="unicode")
    print(field_630_str)

In [115]:
def ead2marc_600_610_630(raw):
    subject_names_list = []
    subject_persnames_list = []
    subject_corpnames_list = []
    subject_famnames_list = []
    subject_titles_list = []
    
    ca_fetch = raw.xpath(".//*[local-name()='controlaccess']")
    ca_xml = ca_fetch[0]
    for ca_comp in ca_xml:
        subject_names_list.append(ca_comp)
        if "persname" in ca_comp.tag:
            subject_persnames_list.append(ca_comp)
        elif "corpname" in ca_comp.tag:
            subject_corpnames_list.append(ca_comp)
        elif "famname" in ca_comp.tag:
            subject_famnames_list.append(ca_comp)
        elif "title" in ca_comp.tag:
            subject_titles_list.append(ca_comp)
    
    # TODO: ADD SUPPORT FOR OTHER ASPACE AGENT TYPES (SOFTWARE)
    '''Set function-wide variables'''
    if len(names_list) > 0:
        field_600_610_630_xml_list = []
        for target_name_ead in names_list:
            '''Extract actual name element and send to 600, 610, or 630 function'''
            if target_name_ead in subject_persnames_list:
                field_600_xml = ead2marc_600(target_name_ead)
                field_600_610_630_xml_list.append(field_600_xml)
            elif target_name_ead in subject_famnames_list:
                field_600_xml = ead2marc_600(target_name_ead)
                field_600_610_630_xml_list.append(field_600_xml)
            elif target_name_ead in subject_corpnames_list:
                field_610_xml = ead2marc_610(target_name_ead)
                field_600_610_630_xml_list.append(field_610_xml)
            elif target_name_ead in subject_titles_list:
                field_630_xml = ead2marc_630(target_name_ead)
                field_600_610_630_xml_list.append(field_630_xml)

        return field_600_610_630_xml_list

    # TODO: CREATE CASES FOR OTHER EAD NAME ELEMENTS (agencyname, geogname, namegrp)
    

ead2marc_600_610_630(c0_raw)

[]

In [116]:
def ead2marc_650(raw):
    sh_fetch = raw.xpath(".//*[local-name()='subject']")
    if len(sh_fetch) > 0:
        field_650_xml_list = []
        for sh in sh_fetch:
            sh_clean = sh.xpath("string()").strip()
            sh_clean = html.escape(sh_clean)

            authority_raw = sh.get("source")
            if authority_raw == "lcac":
                authority = "cyac"
            else:
                authority = authority_raw

            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2'''
            if authority == "lcsh":
                ind2_650 = "0"
            elif authority == "cyac":
                ind2_650 = "1"
            elif authority == "mesh":
                ind2_650 = "2"
            elif authority == "nal":
                ind2_650 = "3"
            elif authority == "":
                ind2_650 = " "
            elif authority == "cash":
                ind2_650 = "5"
            elif authority == "rvm":
                ind2_650 = "6"
            else:
                ind2_650 = "7"

            field_650_open = f"""<datafield tag="650" ind1=" " ind2="{ind2_650}">"""


            '''SUBFIELD A'''
            a_650 = f"""<subfield code="a">{sh_clean}</subfield>"""

            '''SUBFIELD F'''
            if ind2_650 == "7":
                f_650 = f"""<subfield code="2">{authority}</subfield>"""
            else:
                f_650 = ""

            '''PRINT 650 FIELD'''
            # (This portion of code was troubleshot using Claud Opus 4.5)
            field_650_str_nb = field_650_open + a_650 + f_650 + "</datafield>"
            field_650_xml = etree.fromstring(field_650_str_nb)
            field_650_str = etree.tostring(field_650_xml, pretty_print=True, encoding="unicode")
            print(field_650_str)
            field_650_xml_list.append(field_650_xml)

        return field_650_xml_list

ead2marc_650(c0_raw)

<datafield tag="650" ind1=" " ind2="0">
  <subfield code="a">Orchestral music -- Scores</subfield>
</datafield>



[<Element datafield at 0x197291bda40>]

In [117]:
def ead2marc_655(raw):
    gf_fetch = raw.xpath(".//*[local-name()='genreform']")
    if len(gf_fetch) > 0:
        field_655_xml_list = []
        for gf in gf_fetch:
            gf_clean = gf.xpath("string()").strip()
            gf_clean = html.escape(gf_clean)

            authority_raw = gf.get("source")
            if authority_raw == "lcac":
                authority = "cyac"
            else:
                authority = authority_raw

            '''INDICATORS'''
            '''Indicator 1 is constant (blank)'''
            '''Indicator 2'''
            if authority == "lcsh":
                ind2_655 = "0"
            elif authority == "cyac":
                ind2_655 = "1"
            elif authority == "mesh":
                ind2_655 = "2"
            elif authority == "nal":
                ind2_655 = "3"
            elif authority == "":
                ind2_655 = "4"
            elif authority == "cash":
                ind2_655 = "5"
            elif authority == "rvm":
                ind2_655 = "6"
            else:
                ind2_655 = "7"

            field_655_open = f"""<datafield tag="655" ind1=" " ind2="{ind2_655}">"""


            '''SUBFIELD A'''
            a_655 = f"""<subfield code="a">{gf_clean}</subfield>"""

            '''SUBFIELD F'''
            if ind2_655 == "7":
                f_655 = f"""<subfield code="2">{authority}</subfield>"""
            else:
                f_655 = ""

            '''PRINT 655 FIELD'''
            # (This portion of code was troubleshot using Claud Opus 4.5)
            field_655_str_nb = field_655_open + a_655 + f_655 + "</datafield>"
            field_655_xml = etree.fromstring(field_655_str_nb)
            field_655_str = etree.tostring(field_655_xml, pretty_print=True, encoding="unicode")
            print(field_655_str)
            field_655_xml_list.append(field_655_xml)

        return field_655_xml_list

ead2marc_655(c0_raw)

    # NOTE: Indicators beyond $a and $f are not currently supported

In [118]:
def ead2marc_690(raw_root):
    arch_terms_sp = ["collection ", 
                     "Collection ", 
                     "papers ", 
                     "Papers", 
                     "records ", 
                     "Records ",
                     "manuscripts ", 
                     "Manuscripts ",
                     "mss. ",
                     "Mss. "]
    lsh_fetch = raw_root.xpath(".//*[local-name()='titleproper']")
    if len(lsh_fetch) > 0:
        field_690_xml_list = []
        lsh = lsh_fetch[0]
        lsh_all = lsh.xpath("string()").strip()
        lsh_list = lsh_all.split(", ")
        lsh_cleanish = ", ".join(lsh_list[:-1])
        not_cleaned = True
        for arch_term in arch_terms_sp:
            if arch_term in lsh_cleanish:
                not_cleaned = False
                lsh_cleanish_list = re.split(arch_term, lsh_cleanish)
                lsh_clean_part = lsh_cleanish_list[0]
                lsh_clean = lsh_clean_part + arch_term.strip() + "."
        if not_cleaned:
            if lsh_cleanish.endswith(".") == False:
                lsh_clean = lsh_cleanish + "."
            else:
                lsh_clean = lsh_cleanish
        lsh_clean = html.escape(lsh_clean)

        '''INDICATORS'''
        '''Indicator 1 is constant (blank)'''
        '''Indicator 2 is constant (7)'''

        field_690_open = f"""<datafield tag="690" ind1=" " ind2="7">"""


        '''SUBFIELD A'''
        a_690 = f"""<subfield code="a">{lsh_clean}</subfield>"""

        '''SUBFIELD 2'''
        sf2_690 = f"""<subfield code="2">local</subfield>"""

        '''SUBFIELD 5'''
        sf5_690 = f"""<subfield code="5">InU-Mu</subfield>"""

        '''PRINT 690 FIELD'''
        # (This portion of code was troubleshot using Claud Opus 4.5)
        field_690_str_nb = field_690_open + a_690 + sf2_690 + sf5_690 + "</datafield>"
        field_690_xml = etree.fromstring(field_690_str_nb)
        field_690_str = etree.tostring(field_690_xml, pretty_print=True, encoding="unicode")
        print(field_690_str)
        field_690_xml_list.append(field_690_xml)

        return field_690_xml_list

ead2marc_690(root)

<datafield tag="690" ind1=" " ind2="7">
  <subfield code="a">Alice and Hans Tischler Collection.</subfield>
  <subfield code="2">local</subfield>
  <subfield code="5">InU-Mu</subfield>
</datafield>



[<Element datafield at 0x197291bcb00>]

In [119]:
def ead2marc_700(name):
    '''Check if name is associated with an authority file'''
    # (This portion of code partially revised utilizing ChatGPT 5.2)
    '''Pull identifier'''
    if name.get("source") in {"lcnaf", "naf", "viaf"} and name.get("identifier"):
        authfile_no = name.get("identifier")
    elif name.get("source") in {"lcnaf", "naf"}:
        name_str = name.xpath("string()").strip()
        suggest_url = f"""https://id.loc.gov/authorities/names/suggest2?q={name_str}"""
        # (This portion of code was generated utilizing Claude Opus 4.5)
        suggest_response = requests.get(suggest_url)
        suggest_data = suggest_response.json()
        authfile_no = None
        for hit in suggest_data.get("hits", []):
            if hit["aLabel"] == name_str:
                authfile_no = hit["token"]
                break
    elif name.get("source") == "viaf":
        # (This portion of code was generated utilizing Claude Opus 4.5)
        name_str = name.xpath("string()").strip()
        viaf_search_url = f"""https://viaf.org/viaf/search?query=local.personalNames+all+%22{name_str}%22&sortKeys=holdingscount&maximumRecords=5"""
        viaf_headers = {'Accept': 'application/xml'}
        viaf_search_response = requests.get(viaf_search_url, headers=viaf_headers)
        viaf_search_root = etree.fromstring(viaf_search_response.content)
        authfile_no = None
        records = viaf_search_root.xpath('//*[local-name()="record"]')
        for rec in records:
            headings = rec.xpath('.//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_id = rec.xpath('.//*[local-name()="viafID"]')
            if headings and viaf_id and headings[0].text == name_str:
                authfile_no = viaf_id[0].text
                break
    else:
        authfile_no = None

    '''Pull authority file using identifier'''
    if name.get("source") in {"lcnaf", "naf"} and authfile_no:
        '''Get Library of Congress Name Authority MARC/XML and clean'''
        authority = "lcnaf"
        authfile_no = name.get("identifier")
        authority_url = f"""https://lccn.loc.gov/{authfile_no}/marcxml"""
        authority_xml = requests.get(authority_url).content
        authority_root = etree.fromstring(authority_xml)
        authority_700_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='100']")
        # (This portion of code was generated utilizing Claude Opus 4.6)
        '''Fallback to tag 110 if tag 100 not found (handles mismatched EAD name types)'''
        if not authority_700_list:
            authority_700_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
        authority_700_raw = authority_700_list[0]
        '''Clean authority_700_raw'''
        authority_700_str = etree.tostring(authority_700_raw, pretty_print=True, encoding="unicode")
        authority_700_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_700_str)
        authority_700_str_list = authority_700_str.split("\n")
        authority_700_str_list_stripped = [str.strip() for str in authority_700_str_list]
        authority_700_str = "".join(authority_700_str_list_stripped)
        authority_700_str = re.sub(r'</datafield>', '', authority_700_str).strip()
        '''Change tag from 100 to 700'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority_700_str = authority_700_str.replace('tag="100"', 'tag="700"')
        # (This portion of code was generated utilizing Claude Opus 4.6)
        authority_700_str = authority_700_str.replace('tag="110"', 'tag="710"')
    elif name.get("source") == "viaf" and authfile_no:
        '''Get VIAF cluster XML and extract 700 field data'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority = "viaf"
        viaf_headers = {'Accept': 'application/xml'}
        viaf_url = f"https://viaf.org/viaf/{authfile_no}"
        viaf_response = requests.get(viaf_url, headers=viaf_headers)
        viaf_root = etree.fromstring(viaf_response.content)
        '''Check if VIAF cluster has linked LCNAF -- if so, use LCNAF'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        lc_sources = viaf_root.xpath('//*[local-name()="source" and starts-with(text(), "LC|")]')
        if lc_sources:
            '''VIAF has LC link -- fetch from LCNAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            lc_id = lc_sources[0].text.split('|')[1]
            authority_url = f"https://lccn.loc.gov/{lc_id}/marcxml"
            authority_xml = requests.get(authority_url).content
            authority_root = etree.fromstring(authority_xml)
            authority_700_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='100']")
            # (This portion of code was generated utilizing Claude Opus 4.6)
            '''Fallback to tag 110 if tag 100 not found (handles mismatched EAD name types)'''
            if not authority_700_list:
                authority_700_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
            authority_700_raw = authority_700_list[0]
            authority_700_str = etree.tostring(authority_700_raw, pretty_print=True, encoding="unicode")
            authority_700_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_700_str)
            authority_700_str_list = authority_700_str.split("\n")
            authority_700_str_list_stripped = [str.strip() for str in authority_700_str_list]
            authority_700_str = "".join(authority_700_str_list_stripped)
            authority_700_str = re.sub(r'</datafield>', '', authority_700_str).strip()
            '''Change tag from 100 to 700'''
            authority_700_str = authority_700_str.replace('tag="100"', 'tag="700"')
            # (This portion of code was generated utilizing Claude Opus 4.6)
            authority_700_str = authority_700_str.replace('tag="110"', 'tag="710"')
        else:
            '''No LC source -- parse VIAF cluster directly'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_headings = viaf_root.xpath('//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_main_heading = viaf_headings[0].text if viaf_headings else None
            '''Get normalized dates from VIAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_birth = viaf_root.xpath('//*[local-name()="birthDate"]')
            viaf_death = viaf_root.xpath('//*[local-name()="deathDate"]')
            viaf_birth_year = viaf_birth[0].text[:4] if viaf_birth and viaf_birth[0].text and not viaf_birth[0].text.startswith('0') else None
            viaf_death_year = viaf_death[0].text[:4] if viaf_death and viaf_death[0].text and not viaf_death[0].text.startswith('0') else None
            '''Parse heading to separate name from dates'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_parts = viaf_main_heading.split(', ') if viaf_main_heading else []
            viaf_name_parts = []
            for part in viaf_parts:
                if not (any(c.isdigit() for c in part) and ('-' in part or part.endswith('-'))):
                    viaf_name_parts.append(part)
            viaf_ind1 = '1' if len(viaf_name_parts) > 1 else '0'
            viaf_a_content = ', '.join(viaf_name_parts)
            '''Determine date subfield'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            if viaf_birth_year and viaf_death_year:
                viaf_d_content = f'{viaf_birth_year}-{viaf_death_year}'
            elif viaf_birth_year:
                viaf_d_content = f'{viaf_birth_year}-'
            else:
                viaf_d_content = None
            '''Build authority_700_str for VIAF-direct'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_subfields = f'<subfield code="a">{viaf_a_content}</subfield>'
            if viaf_d_content:
                viaf_subfields += f'<subfield code="d">{viaf_d_content}</subfield>'
            authority_700_str = f'<datafield tag="700" ind1="{viaf_ind1}" ind2=" ">{viaf_subfields}'
    else:
        authority = None

    '''INDICATORS'''
    '''Indicator 1'''
    if authority not in ["lcnaf", "viaf"]:
        a_content = name.xpath("string()").strip()
        a_alpha = []
        d_num = []
        a_split = a_content.split(", ")
        for item in a_split:
            if any(char.isnumeric() for char in item):
                d_num.append(item)
            else:
                a_alpha.append(item)
        if len(a_alpha) > 1:
            ind1_700 = "1"
        else:
            ind1_700 = "0"
    else:
        ind1_700 = ""

    '''Indicator 2 is constant (blank)'''

    '''SUBFIELD A'''
    if authority not in ["lcnaf", "viaf"]:
        a_content = ", ".join(a_alpha)
        a_700 = f"""<subfield code ="a">{a_content}</subfield>"""

    '''SUBFIELD D'''
    if authority not in ["lcnaf", "viaf"] and d_num:
        d_content = d_num[0]
        d_content = d_content.rstrip(".")
        d_700 = f"""<subfield code ="d">{d_content}</subfield>"""
    else:
        d_700 = ""

    '''SUBFIELD E'''
    if 'relator' in name.attrib:
        aspace_relator = name.attrib["relator"].lower()
        if aspace_relator in marc_rda_relators.keys():
            e_content = marc_rda_relators[aspace_relator]
            e_700 = f"""<subfield code ="e">{e_content}</subfield>"""
        else:
            e_700 = ""
    else:
        e_700 = ""

    '''PRINT 700 FIELD'''
    if authority == "lcnaf" or authority == "viaf":
        field_700_str_nb = authority_700_str + e_700 + "</datafield>"
        field_700_xml = etree.fromstring(field_700_str_nb)
        field_700_str = etree.tostring(field_700_xml, pretty_print=True, encoding="unicode")
    else:
        field_700_open = f"""<datafield tag="700" ind1="{ind1_700}" ind2=" ">"""
        field_700_str_nb = field_700_open + a_700 + d_700 + e_700 + "</datafield>"
        field_700_xml = etree.fromstring(field_700_str_nb)
        field_700_str = etree.tostring(field_700_xml, pretty_print=True, encoding="unicode")

    '''Reorder attributes to put tag first'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    field_700_str = re.sub(r'<datafield ind1="(.)" ind2="(.)" tag="(\d+)">', r'<datafield tag="\3" ind1="\1" ind2="\2">', field_700_str)

    print(field_700_str)
    # (This portion of code was generated utilizing Claude Opus 4.6)
    return field_700_xml

In [120]:
def ead2marc_710(name):
    '''Check if main name is associated with an authority file'''
    # (This portion of code partially revised utilizing ChatGPT 5.2)
    '''Pull identifier'''
    if name.get("source") in {"lcnaf", "naf", "viaf"} and name.get("identifier"):
        authfile_no = name.get("identifier")
    elif name.get("source") in {"lcnaf", "naf"}:
        name_str = name.xpath("string()").strip()
        suggest_url = f"""https://id.loc.gov/authorities/names/suggest2?q={name_str}"""
        # (This portion of code was generated utilizing Claude Opus 4.5)
        suggest_response = requests.get(suggest_url)
        suggest_data = suggest_response.json()
        authfile_no = None
        for hit in suggest_data.get("hits", []):
            if hit["aLabel"] == name_str:
                authfile_no = hit["token"]
                break
    elif name.get("source") == "viaf":
        # (This portion of code was generated utilizing Claude Opus 4.5)
        name_str = name.xpath("string()").strip()
        viaf_search_url = f"""https://viaf.org/viaf/search?query=local.corporateNames+all+%22{name_str}%22&sortKeys=holdingscount&maximumRecords=5"""
        viaf_headers = {'Accept': 'application/xml'}
        viaf_search_response = requests.get(viaf_search_url, headers=viaf_headers)
        viaf_search_root = etree.fromstring(viaf_search_response.content)
        authfile_no = None
        records = viaf_search_root.xpath('//*[local-name()="record"]')
        for rec in records:
            headings = rec.xpath('.//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_id = rec.xpath('.//*[local-name()="viafID"]')
            if headings and viaf_id and headings[0].text == name_str:
                authfile_no = viaf_id[0].text
                break
    else:
        authfile_no = None

    '''Pull authority file using identifier'''
    if name.get("source") in {"lcnaf", "naf"} and authfile_no:
        '''Get Library of Congress Name Authority MARC/XML and clean'''
        authority = "lcnaf"
        authority_url = f"""https://lccn.loc.gov/{authfile_no}/marcxml"""
        authority_xml = requests.get(authority_url).content
        authority_root = etree.fromstring(authority_xml)
        authority_710_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
        authority_710_raw = authority_710_list[0]
        '''Clean authority_710_raw'''
        authority_710_str = etree.tostring(authority_710_raw, pretty_print=True, encoding="unicode")
        authority_710_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_710_str)
        authority_710_str_list = authority_710_str.split("\n")
        authority_710_str_list_stripped = [str.strip() for str in authority_710_str_list]
        authority_710_str = "".join(authority_710_str_list_stripped)
        authority_710_str = re.sub(r'</datafield>', '', authority_710_str).strip()
        '''Change tag from 110 to 710'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority_710_str = authority_710_str.replace('tag="110"', 'tag="710"')
    elif name.get("source") == "viaf" and authfile_no:
        '''Get VIAF cluster XML and extract 710 field data'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        authority = "viaf"
        viaf_headers = {'Accept': 'application/xml'}
        viaf_url = f"https://viaf.org/viaf/{authfile_no}"
        viaf_response = requests.get(viaf_url, headers=viaf_headers)
        viaf_root = etree.fromstring(viaf_response.content)
        '''Check if VIAF cluster has linked LCNAF -- if so, use LCNAF'''
        # (This portion of code was generated utilizing Claude Opus 4.5)
        lc_sources = viaf_root.xpath('//*[local-name()="source" and starts-with(text(), "LC|")]')
        if lc_sources:
            '''VIAF has LC link -- fetch from LCNAF'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            lc_id = lc_sources[0].text.split('|')[1]
            authority_url = f"https://lccn.loc.gov/{lc_id}/marcxml"
            authority_xml = requests.get(authority_url).content
            authority_root = etree.fromstring(authority_xml)
            authority_710_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='110']")
            authority_710_raw = authority_710_list[0]
            authority_710_str = etree.tostring(authority_710_raw, pretty_print=True, encoding="unicode")
            authority_710_str = re.sub(r'\s+xmlns(:zs)?="[^"]+"', '', authority_710_str)
            authority_710_str_list = authority_710_str.split("\n")
            authority_710_str_list_stripped = [str.strip() for str in authority_710_str_list]
            authority_710_str = "".join(authority_710_str_list_stripped)
            authority_710_str = re.sub(r'</datafield>', '', authority_710_str).strip()
            '''Change tag from 110 to 710'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            authority_710_str = authority_710_str.replace('tag="110"', 'tag="710"')
        else:
            '''No LC source -- parse VIAF cluster directly'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_headings = viaf_root.xpath('//*[local-name()="mainHeadings"]/*[local-name()="data"]/*[local-name()="text"]')
            viaf_main_heading = viaf_headings[0].text if viaf_headings else None
            '''Build authority_710_str for VIAF-direct'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            viaf_subfields = f'<subfield code="a">{viaf_main_heading}</subfield>'
            authority_710_str = f'<datafield tag="710" ind1="2" ind2=" ">{viaf_subfields}'
    else:
        authority = None

    '''INDICATORS'''
    '''Indicator 1'''
    if authority not in ["lcnaf", "viaf"]:
        ind1_710 = "2"
    # TODO: Add support for ind1 0 (inverted name) and 1 (jurisdiction name)

    '''Indicator 2 is constant (blank)'''

    '''SUBFIELDS'''
    '''SUBFIELD A'''
    if authority not in ["lcnaf", "viaf"]:
        a_content = name.xpath("string()").strip()
        a_710 = f"""<subfield code ="a">{a_content}</subfield>"""

    '''SUBFIELD E'''
    if 'relator' in name.attrib:
        aspace_relator = name.attrib["relator"].lower()
        if aspace_relator in marc_rda_relators.keys():
            e_content = marc_rda_relators[aspace_relator]
            e_710 = f"""<subfield code ="e">{e_content}</subfield>"""
        else:
            e_710 = ""
    else:
        e_710 = ""

    '''PRINT 710 FIELD'''
    if authority == "lcnaf" or authority == "viaf":
        field_710_str_nb = authority_710_str + e_710 + "</datafield>"
        field_710_xml = etree.fromstring(field_710_str_nb)
        field_710_str = etree.tostring(field_710_xml, pretty_print=True, encoding="unicode")
    else:
        field_710_open = f"""<datafield tag="710" ind1="{ind1_710}" ind2=" ">"""
        field_710_str_nb = field_710_open + a_710 + e_710 + "</datafield>"
        field_710_xml = etree.fromstring(field_710_str_nb)
        field_710_str = etree.tostring(field_710_xml, pretty_print=True, encoding="unicode")
    
    '''Reorder attributes to put tag first'''
    # (This portion of code was generated utilizing Claude Opus 4.5)
    field_710_str = re.sub(r'<datafield ind1="(.)" ind2="(.)" tag="(\d+)">', r'<datafield tag="\3" ind1="\1" ind2="\2">', field_710_str)
        
    print(field_710_str)
    return field_710_xml
    
    # NOTE:
        # Currently no subfields beyond subfields A and E are supported for non-authority 710s. All content except relators will be placed in subfield A.

In [121]:
def ead2marc_700_710(names_list):
    '''Set function-wide variables'''
    other_names_ead = names_list[1:]
    
    if len(other_names_ead) > 0:
        field_700_710_xml_list = []
        for target_name_ead in other_names_ead:
            '''Extract actual name element and send to 700 or 710 functions'''
            # (This portion of code was generated utilizing Claude Opus 4.5)
            if target_name_ead in creator_persnames_list:
                name_element = target_name_ead.xpath(".//*[local-name()='persname']")[0]
                field_700_xml = ead2marc_700(name_element)
                field_700_710_xml_list.append(field_700_xml)
            elif target_name_ead in creator_famnames_list:
                name_element = target_name_ead.xpath(".//*[local-name()='famname']")[0]
                field_700_xml = ead2marc_700(name_element)
                field_700_710_xml_list.append(field_700_xml)
            elif target_name_ead in creator_corpnames_list:
                name_element = target_name_ead.xpath(".//*[local-name()='corpname']")[0]
                field_710_xml = ead2marc_710(name_element)
                field_700_710_xml_list.append(field_710_xml)

        return field_700_710_xml_list

    # TODO: CREATE CASES FOR OTHER EAD NAME ELEMENTS (agencyname, geogname, namegrp)
    

ead2marc_700_710(creator_names_list)

In [122]:
def ead2marc_856(raw):
    field_856_xml_list = []
    if vaid_clean:

        '''INDICATORS'''
        '''Indicator 1 is constant (4)'''
        '''Indicator 2 is constant (2)'''


        '''SUBFIELDS'''

        '''SUBFIELD U'''
        faid_uri = f"""https://archives.iu.edu/catalog/{vaid_clean}"""
        u_856 = f"""<subfield code="u">{faid_uri} </subfield>"""

        '''SUBFIELD Z'''
        z_856 = """<subfield code="z">Finding aid</subfield>"""

        
        '''PRINT 856 FIELD'''
        field_856_str_nb = """<datafield tag="856" ind1="4" ind2="2">""" + u_856 + z_856 + "</datafield>"
        field_856_xml = etree.fromstring(field_856_str_nb)
        field_856_xml_list.append(field_856_xml)
        field_856_str = etree.tostring(field_856_xml, pretty_print=True, encoding="unicode")
        
        print(field_856_str)

    return field_856_xml_list


ead2marc_856(c0_raw)

<datafield tag="856" ind1="4" ind2="2">
  <subfield code="u">https://archives.iu.edu/catalog/VAE4896 </subfield>
  <subfield code="z">Finding aid</subfield>
</datafield>



[<Element datafield at 0x1972923c540>]